# 模块概述

WtExeFact 是 WonderTrader 执行单元工厂模块，负责提供各种智能订单执行算法，实现交易订单的智能执行和最优成交。主要包括：
- 执行单元工厂管理和创建
- 多种执行算法实现（TWAP、VWAP、最小冲击等）
- 订单生命周期管理
- 期货和股票市场的差异化处理
- 目标驱动和差量驱动两种执行模式

1. **工厂层**（WtExeFact）：
   - 实现 IExecuterFact 接口，提供执行单元的创建、删除和管理功能
   - 支持标准执行单元、差量执行单元和套利执行单元的创建
   - 提供执行单元的枚举和查询功能
   - 实现执行单元的生命周期管理

2. **执行单元层**（各种 ExecuteUnit 实现）：
   - **TWAP执行单元**（WtTWapExeUnit）：时间加权平均价格算法，将订单在指定时间段内均匀分配执行
   - **VWAP执行单元**（WtVWapExeUnit / WtStockVWapExeUnit）：成交量加权平均价格算法，根据历史成交量分布将订单按比例分配到各个时间段执行
   - **最小冲击执行单元**（WtMinImpactExeUnit / WtStockMinImpactExeUnit）：最小市场冲击算法，通过控制订单价格和数量减少对市场的影响
   - **差量最小冲击执行单元**（WtDiffMinImpactExeUnit）：差量模式的最小冲击算法，用于增量驱动模式

3. **订单管理层**（WtOrdMon）：
   - 订单管理器，跟踪和管理执行单元中的订单状态
   - 支持订单注册、查询、超时检查等功能
   - 使用哈希表存储订单信息，提供O(1)查询性能
   - 支持订单超时自动撤单机制

4. **执行模式分类**：
   - **目标驱动模式**（Target Driven）：将仓位调整到目标值，适用于大多数CTA策略
   - **增量驱动模式**（Delta Driven）：立即执行指定的数量变化，适用于高频交易、做市、抢单等策略
   - **组合/价差驱动模式**（Spread Driven）：将组合头寸调整到目标值，当前版本暂不支持

5. **市场类型分类**：
   - **期货市场执行单元**：WtTWapExeUnit、WtVWapExeUnit、WtMinImpactExeUnit、WtDiffMinImpactExeUnit
   - **股票市场执行单元**：WtStockVWapExeUnit、WtStockMinImpactExeUnit
   - 股票市场执行单元支持股数模式、金额模式、比例模式等多种目标模式
   - 股票市场执行单元支持科创板股票（最小下单200股）和可转债（T+0交易、最小下单10张）的特殊处理

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef factoryClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef unitClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef futureUnitClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef stockUnitClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef diffUnitClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef utilClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IExecuterFact["IExecuterFact<br/>执行单元工厂接口<br/>• 创建执行单元<br/>• 删除执行单元<br/>• 枚举执行单元<br/>• 获取工厂名称"]:::interfaceClass
        ExecuteUnit["ExecuteUnit<br/>执行单元基类<br/>• 初始化接口<br/>• 订单回调接口<br/>• 行情回调接口<br/>• 仓位管理接口<br/>• 通道状态接口"]:::interfaceClass
        ExecuteContext["ExecuteContext<br/>执行上下文<br/>• 合约信息<br/>• 交易时段信息<br/>• 下单接口<br/>• 撤单接口"]:::interfaceClass
    end

    %% 工厂层
    subgraph Factory["工厂层 - 执行单元工厂"]
        direction TB
        WtExeFact["WtExeFact<br/>执行单元工厂<br/>• 创建标准执行单元<br/>• 创建差量执行单元<br/>• 创建套利执行单元<br/>• 删除执行单元<br/>• 枚举执行单元"]:::factoryClass
    end

    %% 标准执行单元层（目标驱动模式）
    subgraph StandardUnits["标准执行单元层 - 目标驱动模式"]
        direction TB
        WtTWapExeUnit["WtTWapExeUnit<br/>TWAP执行单元<br/>• 时间加权平均价格算法<br/>• 时间段均匀分配<br/>• 分批执行<br/>• 尾部时间处理<br/>• 订单超时撤单"]:::futureUnitClass
        WtVWapExeUnit["WtVWapExeUnit<br/>VWAP执行单元（期货）<br/>• 成交量加权平均价格算法<br/>• 成交量分布预测<br/>• 按比例分配<br/>• 分批执行<br/>• 尾部时间处理"]:::futureUnitClass
        WtStockVWapExeUnit["WtStockVWapExeUnit<br/>VWAP执行单元（股票）<br/>• 成交量加权平均价格算法<br/>• 股数/金额/比例模式<br/>• 科创板/可转债支持<br/>• 账户资金管理"]:::stockUnitClass
        WtMinImpactExeUnit["WtMinImpactExeUnit<br/>最小冲击执行单元（期货）<br/>• 最小市场冲击算法<br/>• 价格控制<br/>• 数量控制<br/>• 订单超时撤单<br/>• 按比例或固定数量下单"]:::futureUnitClass
        WtStockMinImpactExeUnit["WtStockMinImpactExeUnit<br/>最小冲击执行单元（股票）<br/>• 最小市场冲击算法<br/>• 股数/金额/比例模式<br/>• 科创板/可转债支持<br/>• 账户资金管理<br/>• 错单检测"]:::stockUnitClass
    end

    %% 差量执行单元层（增量驱动模式）
    subgraph DiffUnits["差量执行单元层 - 增量驱动模式"]
        direction TB
        WtDiffMinImpactExeUnit["WtDiffMinImpactExeUnit<br/>差量最小冲击执行单元<br/>• 差量执行模式<br/>• 立即执行数量变化<br/>• 最小市场冲击算法<br/>• 价格控制<br/>• 数量控制"]:::diffUnitClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 订单管理"]
        direction TB
        WtOrdMon["WtOrdMon<br/>订单管理器<br/>• 订单注册<br/>• 订单查询<br/>• 订单超时检查<br/>• 订单枚举<br/>• 订单清理"]:::utilClass
    end

    %% 继承关系
    WtExeFact -.->|"实现"| IExecuterFact
    WtTWapExeUnit -.->|"继承"| ExecuteUnit
    WtVWapExeUnit -.->|"继承"| ExecuteUnit
    WtStockVWapExeUnit -.->|"继承"| ExecuteUnit
    WtMinImpactExeUnit -.->|"继承"| ExecuteUnit
    WtStockMinImpactExeUnit -.->|"继承"| ExecuteUnit
    WtDiffMinImpactExeUnit -.->|"继承"| ExecuteUnit

    %% 工厂创建关系
    WtExeFact -->|"创建"| WtTWapExeUnit
    WtExeFact -->|"创建"| WtVWapExeUnit
    WtExeFact -->|"创建"| WtStockVWapExeUnit
    WtExeFact -->|"创建"| WtMinImpactExeUnit
    WtExeFact -->|"创建"| WtStockMinImpactExeUnit
    WtExeFact -->|"创建差量单元"| WtDiffMinImpactExeUnit

    %% 执行单元使用订单管理器
    WtTWapExeUnit -->|"使用"| WtOrdMon
    WtVWapExeUnit -->|"使用"| WtOrdMon
    WtStockVWapExeUnit -->|"使用"| WtOrdMon
    WtMinImpactExeUnit -->|"使用"| WtOrdMon
    WtStockMinImpactExeUnit -->|"使用"| WtOrdMon
    WtDiffMinImpactExeUnit -->|"使用"| WtOrdMon

    %% 执行单元使用执行上下文
    WtTWapExeUnit -.->|"使用"| ExecuteContext
    WtVWapExeUnit -.->|"使用"| ExecuteContext
    WtStockVWapExeUnit -.->|"使用"| ExecuteContext
    WtMinImpactExeUnit -.->|"使用"| ExecuteContext
    WtStockMinImpactExeUnit -.->|"使用"| ExecuteContext
    WtDiffMinImpactExeUnit -.->|"使用"| ExecuteContext

    %% 应用样式
    class WtExeFact factoryClass
    class WtTWapExeUnit,WtVWapExeUnit,WtMinImpactExeUnit futureUnitClass
    class WtStockVWapExeUnit,WtStockMinImpactExeUnit stockUnitClass
    class WtDiffMinImpactExeUnit diffUnitClass
    class WtOrdMon utilClass
    class IExecuterFact,ExecuteUnit,ExecuteContext interfaceClass
```

# 执行单元工厂 WtExeFact.h/cpp
```cpp
class WtExeFact : public IExecuterFact
```
继承自IExecuterFact接口，执行单元工厂负责创建和管理各种执行单元（ExeUnit），执行单元用于智能执行交易订单。

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称实现
 * 返回执行单元工厂的名称，用于标识和管理不同的执行单元工厂。
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char* WtExeFact::getName()
{
	return FACT_NAME;
}
```

## 枚举执行单元 enumExeUnit
```cpp
/**
 * @brief 枚举执行单元实现
 * 
 * 枚举工厂中所有可用的执行单元，通过回调函数通知调用者。
 * 该函数会枚举所有标准执行单元和差量执行单元。
 * 
 * @param cb 枚举回调函数，对每个执行单元调用，参数为工厂名称、执行单元名称、是否为最后一个
 */
void WtExeFact::enumExeUnit(FuncEnumUnitCallback cb)
{
	// 枚举标准执行单元
	cb(FACT_NAME, "WtTWapExeUnit", false);  // 时间加权平均价格执行单元，不是最后一个
	cb(FACT_NAME, "WtMinImpactExeUnit", true);  // 最小冲击执行单元（期货），是最后一个
	// 注意：这里只枚举了部分执行单元，其他执行单元（如WtStockMinImpactExeUnit、WtVWapExeUnit等）
	// 可能通过其他方式注册或枚举
}
```

## 创建标准执行单元 createExeUnit
```cpp
/**
 * @brief 创建标准执行单元实现
 * 
 * 根据执行单元名称创建对应的标准执行单元实例。
 * 标准执行单元用于目标驱动模式（Target Driven），将仓位调整到目标值。
 * 
 * @param name 执行单元名称
 * @return ExecuteUnit* 返回创建的执行单元对象指针，如果名称不存在则返回NULL
 */
ExecuteUnit* WtExeFact::createExeUnit(const char* name)
{
	if (strcmp(name, "WtTWapExeUnit") == 0) // 如果名称是时间加权平均价格执行单元
		return new WtTWapExeUnit();
	else if (strcmp(name, "WtMinImpactExeUnit") == 0) // 如果名称是最小冲击执行单元（期货）
		return new WtMinImpactExeUnit();
	else if (strcmp(name, "WtStockMinImpactExeUnit") == 0) // 如果名称是最小冲击执行单元（股票）
		return new WtStockMinImpactExeUnit();
	else if (strcmp(name, "WtVWapExeUnit") == 0) // 如果名称是成交量加权平均价格执行单元（期货）
		return  new WtVWapExeUnit();
	else if (strcmp(name, "WtStockVWapExeUnit") == 0) // 如果名称是成交量加权平均价格执行单元（股票）
		return new WtStockVWapExeUnit();
	return NULL;
}
```

## 创建差量执行单元 createDiffExeUnit
```cpp
/**
 * @brief 创建差量执行单元实现
 * 
 * 根据执行单元名称创建对应的差量执行单元实例。
 * 差量执行单元用于增量驱动模式（Delta Driven），立即执行指定的数量变化。
 * 
 * @param name 执行单元名称
 * @return ExecuteUnit* 返回创建的差量执行单元对象指针，如果名称不存在则返回NULL
 */
ExecuteUnit* WtExeFact::createDiffExeUnit(const char* name)
{
	if (strcmp(name, "WtDiffMinImpactExeUnit") == 0)
		return new WtDiffMinImpactExeUnit();
	return NULL;
}
```

## 创建套利执行单元 createArbiExeUnit
```cpp
/**
 * @brief 创建套利执行单元实现
 * 
 * 根据执行单元名称创建对应的套利执行单元实例。
 * 套利执行单元用于组合/价差驱动模式（Spread Driven），将组合头寸调整到目标值。
 * 
 * 注意：当前版本不支持套利执行单元，该函数始终返回NULL。
 * 
 * @param name 执行单元名称
 * @return ExecuteUnit* 返回创建的套利执行单元对象指针，当前版本始终返回NULL
 */
ExecuteUnit* WtExeFact::createArbiExeUnit(const char* name)
{
	return NULL;
}
```

## 删除执行单元 deleteExeUnit
```cpp
/**
 * @brief 删除执行单元实现
 * 
 * 删除指定的执行单元实例，释放其占用的内存。
 * 该函数会检查执行单元是否属于本工厂，只有属于本工厂的执行单元才会被删除。
 * 
 * @param unit 执行单元对象指针
 * @return bool 返回是否删除成功，true表示删除成功，false表示删除失败（不属于本工厂或unit为NULL）
 */
bool WtExeFact::deleteExeUnit(ExecuteUnit* unit)
{
	if (unit == NULL)
		return true;

	if (strcmp(unit->getFactName(), FACT_NAME) != 0) // 如果执行单元不属于本工厂
		return false;

	delete unit; // 删除执行单元实例
	return true;
}
```

# 执行单元层

## 时间加权平均价格执行单元 WtTWapExeUnit.h/cpp
```cpp
class WtTWapExeUnit : public ExecuteUnit
```
TWAP 执行单元将订单在指定时间段内均匀分配执行，以实现接近时间加权平均价格的效果。

### 设计思想
核心在于 **均匀切分时间** 和 **均匀切分数量**。

TWAP 的本质是把一个大单子（例如 1000 手），按照设定的总时间（例如 1 小时），切成无数个小单子（例如 10 次），每隔一段时间（6 分钟）执行一次。
- 在初始化函数 `init` 中，计算了**发单间隔** (`_fire_span`)。
    ```cpp
    // _total_secs: 总时长 (秒)
    // _tail_secs: 收尾时间 (秒，最后预留一段时间专门用来兜底扫单)
    // _total_times: 总共分多少次发
    _fire_span = (_total_secs - _tail_secs) / _total_times; 
    ```
  * 比如要在 10:00 到 10:30 执行，总时长 1800 秒。设了尾部时间 0 秒，分 10 次发。那么 `_fire_span` = 180 秒。程序就知道：**“每过 3 分钟，要醒来干一次活。”**
- 在计算函数 `do_calc` 中，计算了**当前这次应该发多少** (`curQty`)
    ```cpp
    // leftTimes: 剩余还没执行的次数
    // diffQty: 剩余还需要买/卖的总量

    // 核心公式：剩余总量 / 剩余次数
    curQty = std::max(_min_open_lots, round(abs(diffQty) / leftTimes)) * abs(diffQty) / diffQty;
    ```
  * **现实意义**：假设总共要买 1000 手，分 10 次。
    * 第 1 次：剩 1000 手，剩 10 次。计算 `1000 / 10 = 100` 手。下单 100 手。
    * 第 2 次：剩 900 手，剩 9 次。计算 `900 / 9 = 100` 手。下单 100 手。
    * ...
- 代码在 `on_tick`（每收到一笔行情）中检查时间，充当了“闹钟”的角色。
    ```cpp
    // now: 当前时间
    // _last_fire_time: 上次动手的时间
    // 如果距离上次动手已经超过了间隔时间 (_fire_span)
    if (!hasCancel && (now - _last_fire_time >= _fire_span * 1000))
    {
        do_calc();  // 醒醒，该干活了（触发下单逻辑）
    }
    ```

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _target_pos`：目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）
  - `bool _channel_ready`：交易通道是否就绪标志，true表示通道就绪可以下单，false表示通道未就绪

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **执行参数**
  - `uint32_t _total_secs`：执行总时间（单位：秒），从开始时间到结束时间的总时长
  - `uint32_t _total_times`：总执行次数，将订单分成多少批执行
  - `uint32_t _tail_secs`：执行尾部时间（单位：秒），在最后一段时间内集中执行剩余订单
  - `uint32_t _ord_sticky`：挂单时限（单位：秒），订单挂单后超过此时间未成交则自动撤单
  - `uint32_t _price_mode`：价格模式：0-最新价，1-最优价，2-对手价
  - `uint32_t _price_offset`：挂单价格偏移（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `uint32_t _begin_time`：开始时间（格式：HHMM，如1000表示10:00）
  - `uint32_t _end_time`：结束时间（格式：HHMM，如1030表示10:30）
  - `double _min_open_lots`：最小开仓数量（开仓时，如果差量小于此值则不执行）
  - `double _order_lots`：单次发单手数（每次下单的数量）
  - `bool isCanCancel`：是否可撤单标志，true表示可撤单，false表示不可撤单（如涨跌停价的挂单）

- **临时变量**
  - `double _this_target`：本轮目标仓位（当前执行周期的目标仓位）
  - `uint32_t _fire_span`：发单间隔（单位：毫秒），两次下单之间的时间间隔（总时间-尾部时间）/总执行次数
  - `uint32_t _fired_times`：已执行次数（当前已执行的批次数量）
  - `uint64_t _last_fire_time`：上次已执行的时间（毫秒时间戳，用于判断是否到了下次执行时间）
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基本属性

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 返回创建该执行单元的工厂名称。
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char* WtTWapExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 返回执行单元的名称，用于标识和管理。
 * @return const char* 返回执行单元名称字符串（"WtTWapExeUnit"）
 */
const char* WtTWapExeUnit::getName()
{
	return "WtTWapExeUnit";
}
```

#### 初始化执行单元 init
```cpp
/**
 * @brief 初始化执行单元实现
 * 
 * 初始化执行单元，加载配置参数，获取合约信息和交易时段信息。
 * 
 * @param ctx 执行单元运行环境指针，提供合约信息、交易时段信息等
 * @param stdCode 管理的合约代码（标准格式）
 * @param cfg 配置对象指针，包含执行单元的各种配置参数：
 *   - ord_sticky：挂单时限（单位：秒），订单挂单后超过此时间未成交则自动撤单
 *   - begin_time：开始时间（格式：HHMM，如1000表示10:00）
 *   - end_time：结束时间（格式：HHMM，如1030表示10:30）
 *   - total_secs：执行总时间（单位：秒），从开始时间到结束时间的总时长（可选，如果不提供则根据begin_time和end_time计算）
 *   - tail_secs：执行尾部时间（单位：秒），在最后一段时间内集中执行剩余订单
 *   - total_times：总执行次数，将订单分成多少批执行
 *   - price_mode：价格模式（0-最新价，1-最优价，2-对手价）
 *   - price_offset：挂单价格偏移（相对于基准价格的偏移，买入+偏移，卖出-偏移）
 *   - lots：单次发单手数（每次下单的数量）
 *   - minopenlots：最小开仓数量（可选，默认1）
 */
void WtTWapExeUnit::init(ExecuteContext* ctx, const char* stdCode, WTSVariant* cfg)
```

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
负责处理订单状态回报（如订单成交、订单撤销）。它是 TWAP 算法中**订单生命周期管理**的关键环节，特别是负责处理 **撤单后的补单（追单）** 逻辑，确保在本轮分批任务中未完成的数量能够被继续执行。
- 收到撤单通知 -> 划掉旧单子 -> 核对当前进度 -> 发现没达标 -> 马上加价重发，直到达标为止

过程：
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
    * 记录日志。
* **成交重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格试探成功了，后续可以恢复正常逻辑。
* **撤单后补单（核心逻辑，发现之前的挂单超时未成交，并成功将其撤销之后，负责加价重来）**：
  * 如果订单 **已撤销** (`isCanceled`) 且 **所有在途撤单都已回调** (`_cancel_cnt == 0`)：
    * 获取当前真实持仓 `realPos`。
    * 对比 `realPos` 与本轮目标仓位 `_this_target`。
    * **如果未达到目标**（`realPos != _this_target`）：
      * 增加 `_cancel_times`（该计数器会影响后续补单的价格偏移，即追单力度）。
      * 计算差额，调用 `fire_at_once` 立即补单。补单数量为 `max(最小开仓量, 剩余差额)`。
      * *目的：TWAP 的某一笔分批单因为超时或其他原因被撤销了，必须立即补上，不能等到下一个时间窗口。*
* **异常检查**：
  * 如果收到撤单回报但 `_cancel_cnt != 0`，记录错误日志（理论上应等待所有并发撤单完成后再处理）。
```cpp
/**
 * @brief 订单回报处理实现
 * 
 * 当订单状态发生变化时调用此函数，更新订单状态，处理撤单等操作。
 * 
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtTWapExeUnit::on_order(uint32_t localid, const char* stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是 TWAP 策略的**核心驱动器**。它响应行情变化，负责两件事：一是**检查挂单是否超时**并触发撤单；二是**判断时间间隔**，如果到了下一个时间切片，则触发算法计算 (`do_calc`) 进行下单。
* **数据预处理**：
  * 验证 Tick 数据有效性及合约代码匹配。
  * 更新 `_last_tick` 指针（引用计数管理：释放旧的，保留新的）。
* **首笔 Tick 初始化逻辑**（当 `isFirstTick` 为真）：
  * 检查当前时间是否在交易时段内（过滤集合竞价等非连续交易时间）。
  * 检查当前仓位与目标仓位是否一致。如果不一致，立即调用 `do_calc()` 启动算法（不等待时间间隔，立即开始执行）。
* **后续 Tick 运行逻辑**（常态运行）：
    * **挂单超时检查 (TTL)**：
      * 遍历 `_orders_mon` 中的所有活动订单。
      * 如果 `当前时间 - 订单生成时间 > _ord_sticky`（挂单时限）：
        * 调用 `_ctx->cancel` 撤单。
        * 增加 `_cancel_cnt` 计数。
        * 标记 `hasCancel = true`。
  * **时间触发下单**：
    * 如果 **没有触发撤单** (`!hasCancel`) 且 **时间间隔满足** (`now - _last_fire_time >= _fire_span`)：
      * 调用 `do_calc()` 执行核心计算逻辑。
      * *注意：如果刚刚触发了超时撤单，会跳过本次计算，等待 `on_order` 里的撤单回调处理完后再由后续逻辑接管，避免状态冲突。*
```cpp
/**
 * @brief Tick数据回调实现
 * 
 * 当有新的Tick数据时调用此函数，更新最新行情，触发执行计算。
 * 该函数会检查订单超时，自动撤单，然后根据TWAP算法触发执行计算。
 * 
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtTWapExeUnit::on_tick(WTSTickData* newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新持仓和未完成订单数量。
 * 注意：该函数不触发重新计算，因为成交回报会在on_tick中触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtTWapExeUnit::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price)
{
	// 不用触发，这里在ontick里触发（成交回报会在on_tick中触发重新计算）
}
```

#### 下单结果回报处理 on_entrust
处理**发单请求的即时反馈**（同步或准同步回报）。主要用于处理**发单失败**（如拒单、废单）的情况，确保策略状态能及时回滚或重试。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
    * **立即触发重算**：调用 `do_calc()`。
    * *目的：发单失败意味着本该发出去的量没发出去，需要立即重新评估并再次尝试下单，而不是等待下一个时间窗口*
```cpp
/**
 * @brief 下单结果回报处理实现
 * 
 * 当下单请求收到回报时调用此函数，处理下单成功或失败的情况。
 * 如果下单失败，则从订单管理器中删除该订单，并触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtTWapExeUnit::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的目标仓位 set_position
策略的**外部控制接口**。当上层策略决定改变目标仓位时调用此函数。它负责更新目标，并**重置 TWAP 的执行进度**，使其针对新的目标重新开始分批。
* **过滤与去重**：
  * 检查 stdCode 是否与 `_code` 匹配：不匹配则直接返回
  * 如果 `newVol` 与当前 `_target_pos` 相等：直接返回
* **状态更新**：
  * 更新 `_target_pos = newVol`。
* **重置进度**：
  * **关键步骤**：将 `_fired_times`（已执行次数）重置为 0。
  * *含义：无论之前执行到第几步，一旦目标仓位改变，TWAP 算法就将其视为一个新的任务，重新开始计算分批逻辑（重新划分剩余时间和批次）。*
* **触发执行**：
  * 调用 `do_calc()`，立即开始处理新的目标仓位。

```cpp
/**
 * @brief 设置新的目标仓位
 * 
 * 设置执行单元的目标仓位，执行单元会根据TWAP算法在指定时间段内均匀执行。
 * 
 * @param stdCode 合约代码
 * @param newVol 新的目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）
 */
void WtTWapExeUnit::set_position(const char* stdCode, double newVol)
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
当交易通道（柜台）连接成功并就绪时触发的回调。主要用于**状态对齐**和**启动算法**。它负责检查本地策略状态与柜台实际挂单状态是否一致，处理异常订单，并触发第一次计算。
* **标记通道就绪**：
  * 将 `_channel_ready` 设为 `true`，允许后续下单操作。
* **状态检查与对齐**：
  * 获取 `_ctx` 的未完成单数量 (`undone`) 和本地订单管理器的状态 (`_orders_mon`)。
  * **情况 A：柜台有单，本地无单** (`undone != 0 && !_orders_mon.has_order()`)
    * 这属于“不受管”的外部订单或僵尸单。
    * **操作**：
      * 记录日志
      * 根据方向调用 `_ctx->cancel` 强制撤销所有未完成单
      * 将这些撤单操作加入本地监控 (`_orders_mon`)，增加在途撤单计数 (`_cancel_cnt`)。
  * **情况 B：柜台无单，本地有单** (`undone == 0 && _orders_mon.has_order()`)
    * 这通常发生在断线重连后，本地认为有单在挂，但实际上并未发送成功或已被柜台取消。
    * **操作**：为了防止后续逻辑（如超时撤单）出错，
      * 强制清空本地订单管理器 `_orders_mon` 的所有记录
  * **情况 C：状态正常或未知**
    * 记录日志，不做特殊处理。
* **触发计算**：
  * 调用 `do_calc()`，尝试开始第一轮的策略执行。
```cpp
/**
 * @brief 交易通道就绪回调实现
 */
void WtTWapExeUnit::on_channel_ready()
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtTWapExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
**TWAP 策略的核心驱动引擎**。负责根据时间进度计算本轮次应该执行多少量，控制发单频率，并处理清仓逻辑。
* **并发控制（线程安全）**：
  * 使用 `CalcFlag`（原子操作）防止重入（Re-entry）。如果有另一个线程正在执行 `do_calc`，当前调用直接返回。
  * 使用 `StdUniqueLock` 锁住 `_mtx_calc`，因为目标仓位修改 (`set_position`) 和行情推送 (`on_tick`) 可能来自不同线程。
* **前置状态校验**：
  * **通道检查**：若通道未就绪 `!_channel_ready`，退出。
  * **撤单检查**：若有在途撤单 (`_cancel_cnt != 0`)，暂停计算，等待撤单回报。
  * **目标检查**：若目标差量 `diffQty = _target_pos - _ctx->getPosition(code)` 为 0，退出。
* **反向/未完成单处理**：
  * **反向单**：如果当前有未完成挂单 (`undone`) 且方向与新目标相反（例如想买，但有未成交的卖单）
    * 调用 `_ctx->cancel` 进行撤单
    * 如果撤单成功：`_orders_mon` 记录撤单ID、`_cancel_cnt` 增加在途撤单量
    * 返回
  * **同向单**：有未完成挂单 (`undone`) 
    * 返回，即停止本轮执行（直到上一轮完结）
* **清仓/完成逻辑**
  * 如果当前仓位 `_ctx->getPosition(code)` 等于 get_real_target(`_target_pos`)：
    * 若不是清仓指令：直接退出。
    * 若是清仓指令 (`_target_pos` 为 DBL_MAX 并且 `_ctx->getPosition(code)` 为 0)
      * 如果多头持仓 `_ctx->getPosition(code, true, 1)` 为 0
        * 返回
      * 否则（由于精度问题残留的 *碎股*）
        * 须设立目标仓位 `newVol = -min(_ctx->getPosition(code, true, 1), _order_lots)` （反向发单进行清理）
* **行情时间过滤**：
  * 比较当前 Tick 时间与 `_last_tick_time`。如果是旧行情（时间戳未更新），不执行，防止重复计算。
* **核心拆单计算（TWAP算法）**：
  * **计算剩余轮次**：`leftTimes = _total_times - _fired_times`。
  * **确定本轮下单量 (`curQty`)**：
    * **情况 A（尾部/最后一次）**：如果 `leftTimes == 0`，触发 **ShowHand（梭哈）模式**。
      * `curQty` = 剩余全部差量（至少为最小开仓量 `_min_open_lots`）。
      * 标记 `bNeedShowHand = true`。
    * **情况 B（正常轮次）**：
      * `curQty = round(剩余总量 / 剩余次数)`。
      * 确保单笔不小于 `_min_open_lots`。
  * **计算目标仓位**：更新 `_this_target = _ctx->getPosition(code) + curQty`。
* **价格计算与修正**：
  * 基础价格：根据 `_price_mode` 取 Latest/Bid/Ask。
  * **扫单加价**：
    * 如果是 **ShowHand 模式**（为了保证最后一定要成交），在基础价上**偏移 5 跳**（跳：最小价格变动单位）。
    * 否则，应用常规配置的 `_price_offset`。
  * **零价修正**：如果计算价格为 0，使用昨收盘价或最新价兜底。
  * **合规性修正（涨跌停保护）**：
    * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
      * 修正为涨停价
      * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
    * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
      * 修正为跌停价
      * 同样禁止撤单。
* **执行下单**：
  * 调用 `_ctx->buy/sell(code, targetPx, abs(curQty))` 发出指令。
  * 更新监控：`_orders_mon.push_order`。
  * 更新状态：`_last_fire_time`（上次发单时间）、`_fired_times`（已发单次数+1）。
  
```cpp
/**
 * @brief 执行计算实现
 * 核心计算函数，根据TWAP算法计算当前应该执行的订单数量，并在指定时间段内均匀分配执行。
 */
void WtTWapExeUnit::do_calc()
```

#### 立即执行指定数量 fire_at_once
**立即执行**指定数量的下单操作。这个函数通常不用于常规的 TWAP 轮次（`do_calc` 有自己的下单逻辑），而是用于**异常恢复**或**追单**场景。例如，当一个订单因为超时被撤销后，需要立即补发剩余数量时调用此函数。
* **前置检查**：
  * 若请求数量 `qty` 为 0，直接返回。
- 锁定并保留当前的 Tick 数据 `_last_tick` (`retain`)。
* **价格计算（核心）**：
  * 根据配置的 `_price_mode` 以及 `_last_tick` 确定基准价：
    * `0`: 最新价 (LastPrice)
    * `1`: 对手价 (Buy用Ask1，Sell用Bid1) — *注：代码注释写的是最优价，但逻辑通常对应挂单方式，需结合上下文，此处代码为 Buy取Bid(本方最优)，Sell取Ask。*
    * `2`: 激进价 (Buy用Ask1，Sell用Bid1) — *即吃单价格。*
  * **追单偏移**：
    * `targetPx += _comm_info->getPriceTick() * _cancel_times * (isBuy ? 1 : -1)`
    * 利用 `_cancel_times`（撤单次数）作为乘数。如果订单反复被撤（说明挂单挂不进去），会随着撤单次数增加而不断提高买入价（或降低卖出价），以提高成交概率。
* **合规性修正（涨跌停保护）**：
  * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
    * 修正为涨停价
    * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
  * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
    * 修正为跌停价
    * 同样禁止撤单。
* **执行下单**：
  * 根据 `qty` 正负调用 `_ctx->buy` 或 `_ctx->sell`。
  * 将生成的订单 ID 组推入订单监控 (`_orders_mon`)，传入 `isCanCancel` 标志。
* **资源释放**：
  * 释放 Tick 数据引用。

```cpp
/**
 * @brief 立即执行指定数量实现
 * 立即下单执行指定数量的订单，用于尾部时间集中执行剩余订单或撤单后重新下单。
 * @param qty 要执行的订单数量（正数表示买入，负数表示卖出）
 */
void WtTWapExeUnit::fire_at_once(double qty)
```

#### 涨跌停保护
这里代码中的 “涨停板挂单不可撤” (`isCanCancel = false`)，是一个**为了保护排队优势**的策略逻辑。

在 TWAP 等算法交易中，通常有一个**超时撤单机制**
* **正常情况**：如果一个订单挂出去  秒还没成交，算法会认为这个价格不合适或者市场变了，于是撤单，重新计算价格再挂（为了追求成交）。
* **特殊情况**：当买入价触及**涨停板**（或卖出价触及跌停板）时，程序强制将 `isCanCancel` 设为 `false`。这意味着：**即使在这个订单上挂了很久没成交，算法也不允许自动撤单，必须一直挂着死等。**

股票交易遵循“价格优先，时间优先”的原则。

* **价格优先**：出价高的人先买到。
* **时间优先**：出价相同时，先挂单的人先买到。

当股票**涨停**时，价格已经到了交易所允许的最高点
* 此时，所有想买的人都只能出这个最高价，无法出更高的价格。

如果算法不加这个保护逻辑，会发生什么？
1. 以涨停价挂入买单，排在队伍的第 100 位。
2. 过了 10 秒没成交，算法触发“超时撤单”逻辑，把单子撤了。
3. 算法重新计算，发现还是只能以涨停价买入，于是立刻又挂了一个新单。
4. **后果**：你的新单子会排在队伍的**最后一位**（比如第 10000 位）。原本可能快轮到你了，结果你因为“撤单重挂”，自己放弃了位置去重新排队。

## 成交量加权平均价格执行单元 WtVWapExeUnit.h/cpp
```cpp
class WtVWapExeUnit : public ExecuteUnit
```

### 设计思想
**成交量加权** 的核心在于：拿着一张 *历史成交量地图*，根据市场活跃度的预测曲线来动态调整下单速度。
- 这个预测曲线就是代码中的 `VwapAim` 数组

初始化 (`init`) 时，程序会读取一个外部文件（例如 `Vwap_rb2310.txt`），这个文件里存放了基于历史数据计算好的**每一分钟应该达到的累计持仓量**。
```cpp
// WtVWapExeUnit.cpp :: init
// 读取 Vwap_xxx.txt 文件
while (getline(s, prz, ',')) {
    // 将文件中的数字存入 VwapAim 数组
    // VwapAim[0] 代表 9:31 应该完成多少，VwapAim[100] 代表 11:10 应该完成多少...
    VwapAim.push_back(stod(prz));
}
```
在计算逻辑 (`do_calc`) 中，程序首先要根据当前时间，算出自己处于交易时段内的第几分钟
```cpp
// WtVWapExeUnit.cpp :: do_calc
// 计算当前时间是今天的第几分钟（InminsTm）
double InminsTm = calTmStamp(_last_tick->actiontime()); 
// 查表：在这个时刻，按照计划我应该已经买了多少了？
double aimQty = VwapAim[InminsTm];
```
然后对比 *计划量* 和 *实际量*，决定这把下多少单。
```cpp
// WtVWapExeUnit.cpp :: do_calc
// 目标预测量 - 当前实际仓位 = 还需要补多少
_Vwap_vol = aimQty - curPos;
// 下单量 curQty (通常就是 _Vwap_vol，会做一些最小手数的取整处理)
curQty = max(_Vwap_vol, _min_open_lots) * ...;
```

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _target_pos`：目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）
  - `bool _channel_ready`：交易通道是否就绪标志，true表示通道就绪可以下单，false表示通道未就绪

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）

- **VWAP算法数据**
  - `vector<double> VwapAim`：分钟记，目标VWAP预测总报单量（数组，每个元素表示一个时间段的预测成交量比例）

- **执行参数**
  - `uint32_t _total_secs`：执行总时间（单位：秒），从开始时间到结束时间的总时长
  - `uint32_t _total_times`：总执行次数，将订单分成多少批执行
  - `uint32_t _tail_secs`：执行尾部时间（单位：秒），在最后一段时间内集中执行剩余订单
  - `uint32_t _ord_sticky`：挂单时限（单位：秒），订单挂单后超过此时间未成交则自动撤单
  - `uint32_t _price_mode`：价格模式：0-最新价，1-最优价，2-对手价
  - `uint32_t _price_offset`：挂单价格偏移（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `uint32_t _begin_time`：开始时间（格式：HHMM，如1000表示10:00）
  - `uint32_t _end_time`：结束时间（格式：HHMM，如1030表示10:30）
  - `double _min_open_lots`：最小开仓数量（开仓时，如果差量小于此值则不执行）
  - `double _order_lots`：单次发单手数（每次下单的数量）
  - `bool isCanCancel`：是否可撤单标志，true表示可撤单，false表示不可撤单（如涨跌停价的挂单）

- **临时变量**
  - `double _this_target`：本轮目标仓位（当前执行周期的目标仓位）
  - `uint32_t _fire_span`：发单间隔（单位：毫秒），两次下单之间的时间间隔（总时间-尾部时间）/总执行次数
  - `uint32_t _fired_times`：已执行次数（当前已执行的批次数量）
  - `uint64_t _last_fire_time`：上次已执行的时间（毫秒时间戳，用于判断是否到了下次执行时间）
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）
  - `double _Vwap_vol`：VWAP单位时间下单量（根据成交量分布计算的当前时间段应该下单的数量）
  - `double _Vwap_prz`：VWAP价格（根据VWAP算法计算的目标价格）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基本属性

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 返回创建该执行单元的工厂名称。
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char * WtVWapExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 返回执行单元的名称，用于标识和管理。
 * @return const char* 返回执行单元名称字符串（"WtVWapExeUnit"）
 */
const char * WtVWapExeUnit::getName()
{
	return "WtVWapExeUnit";
}
```

#### 初始化执行单元 init

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
这是**订单状态回报**的入口。当发出的订单发生状态变化（如部分成交、全部成交、已撤销）时被调用。其核心作用是**维护本地订单状态**以及处理**异常撤单后的补单（追单）逻辑**。

过程：
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
    * 记录日志。
* **成交重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格试探成功了，后续可以恢复正常逻辑。
* **撤单后补单（核心逻辑，发现之前的挂单超时未成交，并成功将其撤销之后，负责加价重来）**：
  * 如果订单 **已撤销** (`isCanceled`) 且 **所有在途撤单都已回调** (`_cancel_cnt == 0`)：
    * 获取当前真实持仓 `realPos`。
    * 对比 `realPos` 与本轮目标仓位 `_this_target`。
    * **如果未达到目标**（`realPos != _this_target`）：
      * 增加 `_cancel_times`（该计数器会影响后续补单的价格偏移，即追单力度）。
      * 计算差额，调用 `fire_at_once` 立即补单。补单数量为 `max(最小开仓量, 剩余差额)`。
      * *目的：TWAP 的某一笔分批单因为超时或其他原因被撤销了，必须立即补上，不能等到下一个时间窗口。*
* **异常检查**：
  * 如果收到撤单回报但 `_cancel_cnt != 0`，记录错误日志（理论上应等待所有并发撤单完成后再处理）。
```cpp
/**
 * @brief 订单回报处理实现
 * 
 * 当订单状态发生变化时调用此函数，更新订单状态，处理撤单等操作。
 * 
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtVWapExeUnit::on_order(uint32_t localid, const char * stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是算法的**驱动引擎（Heartbeat）**。它由行情数据驱动，负责检查时间进度、触发算法执行、以及监控挂单是否超时。
* **数据预处理**：
  * 验证 Tick 数据有效性及合约代码匹配。
  * 更新 `_last_tick` 指针（引用计数管理：释放旧的，保留新的）。
* **首笔 Tick 初始化逻辑**（当 `isFirstTick` 为真）：
  * 检查当前时间是否在交易时段内（过滤集合竞价等非连续交易时间）。
  * 检查当前仓位与目标仓位是否一致。如果不一致，立即调用 `do_calc()` 启动算法（不等待时间间隔，立即开始执行）。
* **后续 Tick 运行逻辑**（常态运行）：
  * **挂单超时检查 (TTL)**：
    * 遍历 `_orders_mon` 中的所有活动订单。
    * 如果 `当前时间 - 订单生成时间 > _ord_sticky`（挂单时限）：
      * 调用 `_ctx->cancel` 撤单。
      * 增加 `_cancel_cnt` 计数。
      * 标记 `hasCancel = true`。
  * **时间触发下单**：
    * 如果 **没有触发撤单** (`!hasCancel`) 且 **时间间隔满足** (`now - _last_fire_time >= _fire_span`)：
      * 调用 `do_calc()` 执行核心计算逻辑。
      * *注意：如果刚刚触发了超时撤单，会跳过本次计算，等待 `on_order` 里的撤单回调处理完后再由后续逻辑接管，避免状态冲突。*
```cpp
/**
 * @brief Tick数据回调实现
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtVWapExeUnit::on_tick(WTSTickData * newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新持仓和未完成订单数量。
 * 注意：该函数不触发重新计算，因为成交回报会在on_tick中触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtVWapExeUnit::on_trade(uint32_t localid, const char * stdCode, bool isBuy, double vol, double price)
{
	// 在ontick中触发（成交回报会在on_tick中触发重新计算）
}
```

#### 下单结果回报处理 on_entrust
处理**下单请求的回报**（即柜台/网关是否收到了下单请求）。注意这与 `on_order`（订单成交/排队状态）不同，这里主要关注下单请求**是否被拒绝**。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
    * **立即触发重算**：调用 `do_calc()`。
    * *目的：发单失败意味着本该发出去的量没发出去，需要立即重新评估并再次尝试下单，而不是等待下一个时间窗口*
```cpp
/**
 * @brief 下单结果回报处理实现
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtVWapExeUnit::on_entrust(uint32_t localid, const char * stdCode, bool bSuccess, const char * message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的目标仓位 set_position
这是执行单元的**指令接收接口**。策略层通过调用此函数告诉执行单元：“你的新目标是多少”。
* **过滤与去重**：
  * 检查 stdCode 是否与 `_code` 匹配：不匹配则直接返回
  * 如果 `newVol` 与当前 `_target_pos` 相等：直接返回
* **状态更新**：
  * 更新 `_target_pos = newVol`。
* **重置进度**：
  * **关键步骤**：将 `_fired_times`（已执行次数）重置为 0。
  * *含义：VWAP 算法依赖于时间切片计数。当目标仓位改变时（例如从买 100 手变成买 200 手），算法视为一个新的任务开始，因此重置“已执行次数”，重新开始按照 VWAP 曲线分配后续的订单量。*
* **触发执行**：
  * 调用 `do_calc()`，立即开始处理新的目标仓位。
```cpp
/**
 * @brief 设置新的目标仓位实现
 * 
 * 设置执行单元的目标仓位，执行单元会根据VWAP算法在指定时间段内按成交量分布执行。
 * 
 * @param stdCode 合约代码
 * @param newVol 新的目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）
 */
void WtVWapExeUnit::set_position(const char * stdCode, double newVol)
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
当交易通道（柜台）连接成功并就绪时触发的回调。主要用于**状态对齐**和**启动算法**。它负责检查本地策略状态与柜台实际挂单状态是否一致，处理异常订单，并触发第一次计算。
* **标记通道就绪**：
  * 将 `_channel_ready` 设为 `true`，允许后续下单操作。
* **状态检查与对齐**：
  * 获取 `_ctx` 的未完成单数量 (`undone`) 和本地订单管理器的状态 (`_orders_mon`)。
  * **情况 A：柜台有单，本地无单** (`undone != 0 && !_orders_mon.has_order()`)
    * 这属于“不受管”的外部订单或僵尸单。
    * **操作**：
      * 记录日志
      * 根据方向调用 `_ctx->cancel` 强制撤销所有未完成单
      * 将这些撤单操作加入本地监控 (`_orders_mon`)，增加在途撤单计数 (`_cancel_cnt`)。
  * **情况 B：柜台无单，本地有单** (`undone == 0 && _orders_mon.has_order()`)
    * 这通常发生在断线重连后，本地认为有单在挂，但实际上并未发送成功或已被柜台取消。
    * **操作**：为了防止后续逻辑（如超时撤单）出错，
      * 强制清空本地订单管理器 `_orders_mon` 的所有记录
  * **情况 C：状态正常或未知**
    * 记录日志，不做特殊处理。
* **触发计算**：
  * 调用 `do_calc()`，尝试开始第一轮的策略执行。
```cpp
/**
 * @brief 交易通道就绪回调实现
 */
void WtVWapExeUnit::on_channel_ready()
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtVWapExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
**VWAP 策略的核心驱动引擎**。负责根据当前时间在 VWAP 曲线中的位置，计算本轮次应该完成的累计目标量，并发出指令。
* **并发控制与锁**
  * 使用 `CalcFlag`（原子操作）防止重入（Re-entry）。
  * 使用 `StdUniqueLock` 锁住 `_mtx_calc`，保证线程安全。
* **前置状态校验**
  * **通道检查**：若通道未就绪 `!_channel_ready`，退出。
  * **撤单检查**：若有在途撤单 (`_cancel_cnt != 0`)，暂停计算，等待撤单回报。
  * **目标检查**：若目标差量 (`diffQty = get_real_target(_target_pos) - _ctx->getPosition(code)`) 为 0，退出。
* **反向/未完成单处理**：
  * **反向单**：如果当前有未完成挂单 (`undone`) 且方向与新目标相反（例如想买，但有未成交的卖单）
    * 调用 `_ctx->cancel` 进行撤单
    * 如果撤单成功：`_orders_mon` 记录撤单ID、`_cancel_cnt` 增加在途撤单量
    * 返回
  * **同向单**：有未完成挂单 (`undone`) 
    * 返回，即停止本轮执行（直到上一轮完结）
* **清仓/完成逻辑**
  * 如果当前仓位 `_ctx->getPosition(code)` 等于 get_real_target(`_target_pos`)：
    * 若不是清仓指令：直接退出。
    * 若是清仓指令 (`_target_pos` 为 DBL_MAX 并且 `_ctx->getPosition(code)` 为 0)
      * 如果多头持仓 `_ctx->getPosition(code, true, 1)` 为 0
        * 返回
      * 否则（由于精度问题残留的 *碎股*）
        * 须设立目标仓位 `newVol = -min(_ctx->getPosition(code, true, 1), _order_lots)` （反向发单进行清理）
* **行情时间过滤**
  * 比较当前 Tick 时间与 `_last_tick_time`。如果是旧行情（时间戳未更新），不执行，防止重复计算。
* **核心拆单计算（VWAP算法）**
  * **时间定位**：调用 `calTmStamp` 计算当前时间 `_last_tick->actiontime()` 对应交易日的第几分钟（索引）。
  * **获取目标量**：从 `VwapAim` 数组（预加载的成交量分布预测曲线）中获取当前时刻应达到的**累计目标量** `aimQty`。
  * **计算本轮量 (`curQty`)**：
    * 计算理论单量：`_Vwap_vol = aimQty -  _ctx->getPosition(code)`。
    * **情况 A（ShowHand/梭哈模式）**（如果剩余次数 `_total_times - _fired_times` 为 0（最后一次执行））
      * curQty =max(diffQty, _min_open_lots)，即强制买入剩余所有差量
      * 并开启激进模式 (`bNeedShowHand = true`)。
    * **情况 B（正常轮次）**：取 `_Vwap_vol` 和 `_min_open_lots` 的最大值，并根据方向调整符号。
  * **更新本轮目标**：`_this_target = _ctx->getPosition(code) + curQty`。
* **价格计算与修正**：
  * 基础价格：根据 `_price_mode` 取 Latest/Bid/Ask。
  * **扫单加价**：
    * 如果是 **ShowHand 模式**（为了保证最后一定要成交），在基础价上**偏移 5 跳**（跳：最小价格变动单位）。
    * 否则，应用常规配置的 `_price_offset`。
  * **零价修正**：如果计算价格为 0，使用昨收盘价或最新价兜底。
  * **合规性修正（涨跌停保护）**：
    * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
      * 修正为涨停价
      * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
    * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
      * 修正为跌停价
      * 同样禁止撤单。
* **执行下单**：
  * 调用 `_ctx->buy/sell(code, targetPx, abs(curQty))` 发出指令。
  * 更新监控：`_orders_mon.push_order`。
  * 更新状态：`_last_fire_time`（上次发单时间）、`_fired_times`（已发单次数+1）。
```cpp
/**
 * @brief 执行计算实现
 */
void WtVWapExeUnit::do_calc()
```

#### 立即执行指定数量 fire_at_once
**立即执行**指定数量的下单操作。这个函数通常用于**异常恢复**或**追单**场景（例如超时撤单后的补单），而不是正常的 VWAP 轮次逻辑。
* **前置检查**：
  * 若请求数量 `qty` 为 0，直接返回。
- 锁定并保留当前的 Tick 数据 `_last_tick` (`retain`)。
* **价格计算（核心）**：
  * 根据配置的 `_price_mode` 以及 `_last_tick` 确定基准价：
    * `0`: 最新价 (LastPrice)
    * `1`: 对手价 (Buy用Ask1，Sell用Bid1) — *注：代码注释写的是最优价，但逻辑通常对应挂单方式，需结合上下文，此处代码为 Buy取Bid(本方最优)，Sell取Ask。*
    * `2`: 激进价 (Buy用Ask1，Sell用Bid1) — *即吃单价格。*
  * **追单偏移**：
    * `targetPx += _comm_info->getPriceTick() * _cancel_times * (isBuy ? 1 : -1)`
    * 利用 `_cancel_times`（撤单次数）作为乘数。如果订单反复被撤（说明挂单挂不进去），会随着撤单次数增加而不断提高买入价（或降低卖出价），以提高成交概率。
* **合规性修正（涨跌停保护）**：
  * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
    * 修正为涨停价
    * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
  * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
    * 修正为跌停价
    * 同样禁止撤单。
* **执行下单**：
  * 根据 `qty` 正负调用 `_ctx->buy` 或 `_ctx->sell`。
  * 将生成的订单 ID 组推入订单监控 (`_orders_mon`)，传入 `isCanCancel` 标志。
* **资源释放**：
  * 释放 Tick 数据引用。
```cpp
/**
 * @brief 立即执行指定数量实现
 * @param qty 要执行的订单数量（正数表示买入，负数表示卖出）
 */
void WtVWapExeUnit::fire_at_once(double qty)
```

## 股票成交量加权平均价格执行单元 WtStockVWapExeUnit.h/cpp
```cpp
class WtStockVWapExeUnit : public ExecuteUnit
```

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _target_pos`：目标仓位（股数模式下的目标股数，正数表示买入，负数表示卖出，DBL_MAX表示清仓）
  - `double _target_amount`：目标金额（金额模式下的目标金额，正数表示买入金额，负数表示卖出金额）
  - `double _avaliable`：账户可用资金（用于金额模式和比例模式的计算）
  - `bool _channel_ready`：交易通道是否就绪标志，true表示通道就绪可以下单，false表示通道未就绪

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）

- **VWAP算法数据**
  - `vector<double> VwapAim`：分钟记，目标VWAP预测总报单量（数组，每个元素表示一个时间段的预测成交量比例）

- **执行参数**
  - `uint32_t _total_secs`：执行总时间（单位：秒），从开始时间到结束时间的总时长
  - `uint32_t _total_times`：总执行次数，将订单分成多少批执行
  - `uint32_t _tail_secs`：执行尾部时间（单位：秒），在最后一段时间内集中执行剩余订单
  - `uint32_t _ord_sticky`：挂单时限（单位：秒），订单挂单后超过此时间未成交则自动撤单
  - `uint32_t _price_mode`：价格模式：0-最新价，1-最优价，2-对手价
  - `uint32_t _price_offset`：挂单价格偏移（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `uint32_t _begin_time`：开始时间（格式：HHMM，如1000表示10:00）
  - `uint32_t _end_time`：结束时间（格式：HHMM，如1030表示10:30）
  - `double _min_open_lots`：最小开仓数量（开仓时，如果差量小于此值则不执行）
  - `double _order_lots`：单次发单手数（每次下单的数量）
  - `bool isCanCancel`：是否可撤单标志，true表示可撤单，false表示不可撤单（如涨跌停价的挂单）

- **股票市场特殊参数**
  - `bool _is_KC`：是否是科创板股票标志（科创板代码>=688000）
  - `TargetMode _target_mode`：目标模式（股数/金额/比例），枚举值：stocks=0（股数模式），amount=1（金额模式），ratio=2（比例模式）
  - `bool _is_clear`：是否清仓标志，true表示正在清仓，false表示不清仓
  - `double _min_hands`：最小手数（根据股票类型自动计算：普通股票100股，科创板200股，可转债10张）
  - `double _start_price`：开始执行时的价格（用于计算执行效果）
  - `double _is_t0`：是否T+0交易标志（对于转债等来说，这个需要是true，股票为false）
  - `bool _is_finish`：是否已完成执行标志，true表示已完成，false表示未完成
  - `uint64_t _start_time`：开始执行时间（毫秒时间戳，用于统计执行时间）

- **临时变量**
  - `double _this_target`：本轮目标仓位（当前执行周期的目标仓位）
  - `uint32_t _fire_span`：发单间隔（单位：毫秒），两次下单之间的时间间隔（总时间-尾部时间）/总执行次数
  - `uint32_t _fired_times`：已执行次数（当前已执行的批次数量）
  - `uint64_t _last_fire_time`：上次已执行的时间（毫秒时间戳，用于判断是否到了下次执行时间）
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）
  - `double _Vwap_vol`：VWAP单位时间下单量（根据成交量分布计算的当前时间段应该下单的数量）
  - `double _Vwap_prz`：VWAP价格（根据VWAP算法计算的目标价格）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基本属性

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 
 * 返回创建该执行单元的工厂名称。
 * 
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char * WtStockVWapExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 返回执行单元的名称，用于标识和管理。
 * @return const char* 返回执行单元名称字符串（"WtStockVWapExeUnit"）
 */
const char * WtStockVWapExeUnit::getName()
{
	return "WtStockVWapExeUnit";
}
```

#### 初始化执行单元 init

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
这是 **订单状态回报** 的入口。当发出的订单发生状态变化（如部分成交、全部成交、已撤销）时被调用。其核心作用是 **维护本地订单状态** 以及处理 **异常撤单后的补单（追单）逻辑**。
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
    * 记录日志。
* **成交重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格试探成功了，后续可以恢复正常逻辑。
* **撤单后补单（核心逻辑，发现之前的挂单超时未成交，并成功将其撤销之后，负责加价重来）**：
  * 如果订单 **已撤销** (`isCanceled`) 且 **所有在途撤单都已回调** (`_cancel_cnt == 0`)：
    * 获取当前真实持仓 `realPos`。
    * 对比 `realPos` 与本轮目标仓位 `_this_target`。
    * **如果未达到目标**（`realPos != _this_target`）：
      * 增加 `_cancel_times`（该计数器会影响后续补单的价格偏移，即追单力度）。
      * 计算差额，调用 `fire_at_once` 立即补单。补单数量为 `max(最小开仓量, 剩余差额)`。
      * *目的：TWAP 的某一笔分批单因为超时或其他原因被撤销了，必须立即补上，不能等到下一个时间窗口。*
* **异常检查**：
  * 如果收到撤单回报但 `_cancel_cnt != 0`，记录错误日志（理论上应等待所有并发撤单完成后再处理）。
```cpp
/**
 * @brief 订单回报处理实现
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtStockVWapExeUnit::on_order(uint32_t localid, const char * stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是算法的 **驱动引擎**。它由行情数据驱动，负责检查时间进度、触发 VWAP 计算、以及监控挂单是否超时。
* **数据预处理**：
  * 验证 Tick 数据有效性及合约代码匹配。
  * 更新 `_last_tick` 指针（引用计数管理：释放旧的，保留新的）。
* **首笔 Tick 初始化逻辑**（当 `isFirstTick` 为真）：
  * 检查当前时间是否在交易时段内（过滤集合竞价等非连续交易时间）。
  * 检查当前仓位与目标仓位是否一致。如果不一致，立即调用 `do_calc()` 启动算法（不等待时间间隔，立即开始执行）。
* **后续 Tick 运行逻辑**（常态运行）：
  * **挂单超时检查 (TTL)**：
    * 遍历 `_orders_mon` 中的所有活动订单。
    * 如果 `当前时间 - 订单生成时间 > _ord_sticky`（挂单时限）：
      * 调用 `_ctx->cancel` 撤单。
      * 增加 `_cancel_cnt` 计数。
      * 标记 `hasCancel = true`。
  * **时间触发下单**：
    * 如果 **没有触发撤单** (`!hasCancel`) 且 **时间间隔满足** (`now - _last_fire_time >= _fire_span`)：
      * 调用 `do_calc()` 执行核心计算逻辑。
      * *注意：如果刚刚触发了超时撤单，会跳过本次计算，等待 `on_order` 里的撤单回调处理完后再由后续逻辑接管，避免状态冲突。*
```cpp
/**
 * @brief Tick数据回调实现
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtStockVWapExeUnit::on_tick(WTSTickData * newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新持仓和未完成订单数量。
 * 注意：该函数不触发重新计算，因为成交回报会在on_tick中触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtStockVWapExeUnit::on_trade(uint32_t localid, const char * stdCode, bool isBuy, double vol, double price)
{
	// 在ontick中触发（成交回报会在on_tick中触发重新计算）
}
```

#### 下单结果回报处理 on_entrust
这是 **下单请求回报** 的处理函数。它关注的是下单请求是否被柜台/网关 **拒绝**（例如废单、资金不足），而不是订单成交状态。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
    * **立即触发重算**：调用 `do_calc()`。
    * *目的：发单失败意味着本该发出去的量没发出去，需要立即重新评估并再次尝试下单，而不是等待下一个时间窗口*
```cpp
/**
 * @brief 下单结果回报处理实现
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtStockVWapExeUnit::on_entrust(uint32_t localid, const char * stdCode, bool bSuccess, const char * message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的目标仓位 set_position
这是执行单元的 **指令接收接口**。策略层通过调用此函数设置新的目标仓位，从而启动或重置执行逻辑。
* **过滤与去重**：
  * 检查 stdCode 是否与 `_code` 匹配：不匹配则直接返回
  * 如果 `newVol` 与当前 `_target_pos` 相等：直接返回
  * 如果 `newVol` 小于 0：返回（股票不支持做空仓位为负）。
* **状态更新**：
  * 更新 `_target_pos = newVol`。
  * 设置目标模式 `_target_mode` 为 `TargetMode::stocks`（股数模式）
  * 重置完成标志 `_is_finish = false`。
  * 开始时间 `_start_time` 为当前本地时间 
  * 开始价格 `_start_price` 为 `_ctx->grabLastTick(_code).price()`
* **重置进度**：
  * 将 `_fired_times`（已执行次数）重置为 0。
* **触发执行**：
  * 调用 `do_calc()`，立即开始处理新的目标仓位。

```cpp
/**
 * @brief 设置新的目标仓位实现
 * @param stdCode 合约代码
 * @param newVol 新的目标仓位（正数表示目标股数，0表示清仓，负数表示错误值）
 */
void WtStockVWapExeUnit::set_position(const char * stdCode, double newVol)
```

#### 清理全部持仓 clear_all_position
```cpp
/**
 * @brief 清理全部持仓实现
 * 
 * 设置执行单元为清仓模式，将目标仓位设置为0，执行单元会将所有持仓卖出。
 * 
 * @param stdCode 合约代码
 */
void WtStockVWapExeUnit::clear_all_position(const char* stdCode) {
	if (_code.compare(stdCode) != 0)  // 如果合约代码不匹配
		return;  // 直接返回，不做任何处理
	_is_clear = true;  // 设置清仓标志为true
	_target_pos = 0;  // 设置目标仓位为0（清仓）
	_target_amount = 0;  // 设置目标金额为0（清仓）
	do_calc();  // 触发执行计算，开始清仓
}
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
当交易通道（柜台）连接成功并就绪时触发。主要用于 **状态对齐**，处理断线重连后的状态不一致问题。
* **标记通道就绪**：
  * 将 `_channel_ready` 设为 `true`，允许后续下单操作。
* **状态检查与对齐**：
  * 获取 `_ctx` 的未完成单数量 (`undone`) 和本地订单管理器的状态 (`_orders_mon`)。
  * **情况 A：柜台有单，本地无单** (`undone != 0 && !_orders_mon.has_order()`)
    * 这属于“不受管”的外部订单或僵尸单。
    * **操作**：
      * 记录日志
      * 根据方向调用 `_ctx->cancel` 强制撤销所有未完成单
      * 将这些撤单操作加入本地监控 (`_orders_mon`)，增加在途撤单计数 (`_cancel_cnt`)。
  * **情况 B：柜台无单，本地有单** (`undone == 0 && _orders_mon.has_order()`)
    * 这通常发生在断线重连后，本地认为有单在挂，但实际上并未发送成功或已被柜台取消。
    * **操作**：为了防止后续逻辑（如超时撤单）出错，
      * 强制清空本地订单管理器 `_orders_mon` 的所有记录
  * **情况 C：状态正常或未知**
    * 记录日志，不做特殊处理。
* **触发计算**：
  * 调用 `do_calc()`，尝试开始第一轮的策略执行。
```cpp
/**
 * @brief 交易通道就绪回调实现
 */
void WtStockVWapExeUnit::on_channel_ready()
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtStockVWapExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
负责在每一个 Tick 到来时决定 *是否下单*、*下多少单* 以及 *下什么价格*
- **并发控制与快照获取**
  1. **原子标志锁**：防止函数重入。
     * 如果上一次 `do_calc` 还没跑完（比如被阻塞），新的调用会直接返回，避免状态错乱。
  2. **互斥锁**：
     * 锁住 `_mtx_calc`，防止在计算过程中外部线程（如 `set_position`）修改了目标仓位。
  3. **获取基础数据快照**：
     * **未完成单 (`undone`)**：当前挂在柜台还没成交的单子 `_ctx->getUndoneQty(code)`
     * **总仓位 (`realPos`)**：`_ctx->getPosition(code)`，昨仓 + 今仓。
     * **可用仓位 (`vailyPos`)**：
       * 如果是 **T+1**（普通股票）：`vailyPos` = 昨仓（今仓不可卖）。
       * 如果是 **T+0**（可转债/债券，`_is_t0=true`）：`vailyPos` = 总仓（今仓可卖）。
     * **目标差量 (`diffQty`)**：用户设定的目标  - 总仓位，`get_real_target(_target_pos) - realPos`
- **T+1 规则修正与合法性校验**
  1. **通道检查**：如果交易通道未就绪 (`!_channel_ready`)，直接退出。
  2. **已完成检查**：如果标记为已完成 (`_is_finish`)，直接退出。
  3. **卖出目标修正（关键细节）**：
     * 操作：`target_pos = max(realPos - vailyPos, _target_pos)`。
     * 逻辑：理论最小持仓是今仓（T+1）或 0（T+0），目标仓位不能低于这个最小持仓
  4. **在途撤单检查**：
     * 如果 `_cancel_cnt != 0`（说明上一轮发起的撤单还没收到柜台回报），直接退出。
     * **必须等待撤单完成才能进行下一步**。
- **完成态判定与方向冲突处理**
  1. **碎股完成判定**：
     * 使用 `round_hands`（按最小手数取整，如 100）对比 `target_pos` 和 `realPos`。
     * **细节**：如果 `round(目标) == round(持仓)`，视为任务完成，标记 `_is_finish = true` 并退出。这容忍了因除权除息或算法取整导致的微小碎股差异。
  2. **反向单处理**：
     * 如果 `diffQty`（想做的方向）与 `undone`（挂单的方向）相反（如想买但有卖单在挂）：
     * **动作**：立即调用 `cancel` 撤单，增加 `_cancel_cnt`，并退出等待下一轮。
  3. **同向单处理**：
     * 如果 `undone != 0`（有同向挂单未成交）：
     * **动作**：退出。**不补单**，耐心等待上一笔挂单成交或超时（由 `on_tick` 的超时逻辑负责撤单）。
- **行情过滤与时间同步** (Tick & Time)
  1. **空数据检查**：如果 `_last_tick` 为空，退出。
  2. **清仓特例检查**：
     * 如果当前仓位 `_ctx->getPosition(code)` 等于 get_real_target(`_target_pos`)：
       * 若不是清仓指令：直接退出。
       * 若是清仓指令 (`_target_pos` 为 DBL_MAX 并且 `_ctx->getPosition(code)` 为 0)
         * 如果多头持仓 `_ctx->getPosition(code, true, 1)` 为 0
           * 返回
         * 否则（由于精度问题残留的 *碎股*）
           * 须设立目标仓位 `newVol = -min(_ctx->getPosition(code, true, 1), _order_lots)` （反向发单进行清理）
  3. **时间戳防抖**：
     * 比较当前 Tick 时间与 `_last_tick_time`。
     * 如果时间没更新（收到重复 Tick），退出，防止重复计算。
- **VWAP 核心量化计算**
  1. **时间定位**：调用 `calTmStamp` 将当前时间转换为交易分钟索引 `InminsTm`。
  2. **查表预测**：读取 `VwapAim[InminsTm]`，获取当前时刻应完成的累计比例/数量 `aimQty`。
  3. **计算理论单量**：`_Vwap_vol = aimQty - curPos`（目标进度 - 当前进度）。
  4. **计算实际单量 (`curQty`)**：
     * **梭哈模式 (ShowHand)**：
       * 条件：`_total_times - _fired_times == 0`（剩余执行次数为 0，即最后一次）。
       * 动作：`curQty = diffQty`（缺多少买多少），并标记 `bNeedShowHand = true`（准备激进报价）。
     * **正常模式**：
       * `curQty = max(_Vwap_vol, _min_open_lots) * (diffQty / abs(diffQty))`。
       * 逻辑：取“VWAP 理论量”和“最小开仓量”的较大值，并赋予正确的买卖符号。
- **手数精密修正**：针对股票市场的特殊规则（买入须整手，卖出可碎股）进行最终修正。
  1. **买入修正**：
     * `curQty = round_hands(curQty, _min_open_lots)`。
     * 强制按 100（或科创板 200）的倍数取整。
  2. **卖出修正**：
     * **碎股特例**：如果 `vailyPos < _min_open_lots`（可用仓位不足一手，比如剩 50 股）：
       * `curQty = vailyPos`（允许直接卖出 50 股，清空碎股）。
     * **正常卖出**：否则，按最小手数取整。
     * **硬限额**：`curQty = min(vailyPos, curQty)`。再次确保卖出量不超过可用昨仓。
  3. **更新本轮目标**：
     * `_this_target = realPos + curQty`（记录这一下去之后的目标，用于 `on_order` 里的撤单补救判断）。
- **价格策略与风控：决定挂单的价格**
  1. **基准价选择**：
     * 根据 `_price_mode`：
       * 0: 最新价 (`LastPrice`)。
       * 1: 最优价 (买用 `Bid1`, 卖用 `Ask1` - *注：代码里买入取 bidprice 可能是保守策略，需结合实际配置确认*).
       * 2: 对手价 (买用 `Ask1`, 卖用 `Bid1`，即吃单价)。
  2. **价格偏移 (Offset)**：
     * 如果是 **ShowHand**（最后一次）：`offset = 5` 跳（买入 +5，卖出 -5），**不惜成本确保成交**。
     * 正常情况：使用配置的 `_price_offset`。
  3. **零价兜底**：
     * 如果计算出的价格是 0（比如开盘前或数据异常），使用昨收价 (`preclose`)。
  4. **合规性修正（涨跌停保护）**
     * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
       * 修正为涨停价
       * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
     * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
       * 修正为跌停价
       * 同样禁止撤单。
* **执行下单**：
  * 调用 `_ctx->buy/sell(code, targetPx, abs(curQty))` 发出指令。
  * 更新监控：`_orders_mon.push_order`。
  * 更新状态：`_last_fire_time`（上次发单时间）、`_fired_times`（已发单次数+1）。
```cpp
/**
 * @brief 执行计算实现
 */
void WtStockVWapExeUnit::do_calc()
```

#### 立即执行指定数量 fire_at_once
这是 **立即执行（补单）** 函数。通常用于异常恢复（如超时撤单后），或者尾部扫尾，不经过 VWAP 曲线计算，直接针对指定数量进行交易。
* **前置检查**：
  * 若请求数量 `qty` 为 0，直接返回。
- 锁定并保留当前的 Tick 数据 `_last_tick` (`retain`)。
* **价格计算（核心）**：
  * 根据配置的 `_price_mode` 以及 `_last_tick` 确定基准价：
    * `0`: 最新价 (LastPrice)
    * `1`: 对手价 (Buy用Ask1，Sell用Bid1) — *注：代码注释写的是最优价，但逻辑通常对应挂单方式，需结合上下文，此处代码为 Buy取Bid(本方最优)，Sell取Ask。*
    * `2`: 激进价 (Buy用Ask1，Sell用Bid1) — *即吃单价格。*
  * **追单偏移**：
    * `targetPx += _comm_info->getPriceTick() * _cancel_times * (isBuy ? 1 : -1)`
    * 利用 `_cancel_times`（撤单次数）作为乘数。如果订单反复被撤（说明挂单挂不进去），会随着撤单次数增加而不断提高买入价（或降低卖出价），以提高成交概率。
* **合规性修正（涨跌停保护）**：
  * 如果买入价 `targetPx` 超过**涨停板** `_last_tick.upper_limit`
    * 修正为涨停价
    * 并标记 `isCanCancel = false`（涨停板挂单不可撤，防止死循环）。
  * 如果卖出价 `targetPx` 低于**跌停板** `_last_tick.lower_limit`
    * 修正为跌停价
    * 同样禁止撤单。
* **执行下单**：
  * 根据 `qty` 正负调用 `_ctx->buy` 或 `_ctx->sell`。
  * 将生成的订单 ID 组推入订单监控 (`_orders_mon`)，传入 `isCanCancel` 标志。
* **资源释放**：
  * 释放 Tick 数据引用。

```cpp
/**
 * @brief 立即执行指定数量实现
 * 
 * 立即下单执行指定数量的订单，用于尾部时间集中执行剩余订单或撤单后重新下单。
 * 
 * @param qty 要执行的订单数量（正数表示买入，负数表示卖出）
 */
void WtStockVWapExeUnit::fire_at_once(double qty)
```

## 最小冲击执行单元 WtMinImpactExeUnit.h/cpp
```cpp
class WtMinImpactExeUnit : public ExecuteUnit
```

### 设计思想
`WtMinImpactExeUnit`（最小冲击执行单元）的设计初衷是为了解决大额订单直接推向市场可能导致的 **滑点** 和 **价格剧烈波动** 问题。

1. 核心策略：逐笔发单
* **原理：** 就像吃一块大蛋糕，一口吞下会被噎住（冲击市场），切成小块慢慢吃就没问题。
* **代码体现：** 在 `do_calc` 函数中，系统首先检查当前是否有未完成的挂单（`undone`）。
  * 如果**有**挂单：系统直接返回，**什么都不做**。必须等上一笔小单完全成交或撤销后，才发下一笔。
  * **效果：** 无论总目标仓位多大（比如要买1000手），市场盘口上永远只看得到一笔很小的单子（比如1手或10手）。这就是典型的“冰山订单”逻辑，隐藏了真实的交易意图。

2. 数量控制：随行就市（动态手数）
* **原理：** 如果市场此刻很冷清，下一大单会把价格砸个坑；如果市场很活跃，就可以多下一点。
* **代码体现：** 下单数量 `this_qty` 有两种计算模式：
  * **固定模式：** 每次只下固定的极小量（由配置 `lots` 决定，如每次1手）。
  * **比例模式（更智能）：** 开启 `_by_rate` 后，它会看对手盘有多少量。比如你要买，它就看“卖一”挂了多少，然后只吃掉其中的一小部分（由配置 `_by_rate` 决定，如10%）。
  * **效果：** 保证每一笔订单的成交量都在市场当前可承受的流动性范围内，不会因为“吃光”盘口而导致价格瞬间跳变。

3. 价格策略：被动等待与温和追单
* **原理：** 尽量不要主动去“扫单”（Taker），而是挂在好的价格等别人来成交（Maker），这样不仅成本低，而且不会推动价格。
* **代码体现：**
  * **自动价格模式（AUTOPX）：** 系统会计算买卖压力 `mp`。如果买盘强，它可能会稍微激进一点；如果势均力敌，它倾向于挂在“买一”或“卖一”等待，而不是直接对价成交。
  * **温和追单：** 如果挂单挂出去一段时间（`_expire_secs`）没人吃，系统会撤单。撤单后，`_cancel_times`（撤单次数）会增加。下一次计算价格时，会在原价基础上**微调**（`PriceTick * _cancel_times`）。
  * **效果：** 就像讨价还价，先报一个对自已有利的价格，对方不答应（超时），再一点点让步，而不是上来就妥协，从而保护了成交均价。
  * **引用：** `buyPx += _comm_info->getPriceTick() * _cancel_times;`

4. 节奏控制：不急不躁（时间间隔）
* **原理：** 连续快速的下单容易被市场中的高频机器人识别并针对。
* **代码体现：** `do_calc` 中会检查当前时间与上次下单时间 `_last_place_time` 的差值。
  * 如果小于设定的间隔 `_entrust_span`（如500毫秒），系统会强制等待，即使条件满足也不下单。
  * **效果：** 强制拉长交易时间线，将冲击力分散到更长的时间维度上。
  * **引用：** `if (now - _last_place_time < _entrust_span) return;`

5. 总结：一个生动的例子
* 假设要买入 **1000手** 某合约：
  1. **WtMinImpactExeUnit** 启动，发现目标差额1000手。
  2. **观察：** 发现卖一价挂着 50手。
  3. **计算：** 设定比例10%，于是只打算买 **5手**。
  4. **定价：** 挂在“买一价”等待（不主动吃卖单，防止推高价格）。
  5. **等待：** 过了10秒（超时时间），这5手还没成交。
  6. **调整：** 撤单，重新计算。这次为了成交，价格提高一跳（追单），发单。
  7. **成交：** 5手成交了。
  8. **休息：** 强制等待 500毫秒（防止被发现）。
  9. **循环：** 检查还剩 995手，重复上述过程。

通过这种**切片 -> 挂单 -> 等待/微调 -> 休息 -> 循环**的过程，该执行单元可以最大程度地维持市场价格的稳定。

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _target_pos`：目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **价格控制参数**
  - `int32_t _price_offset`：价格偏移跳数（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `int32_t _price_mode`：价格模式：0-最新价，-1-最优价，1-对手价，2-自动价格
  - `uint32_t _expire_secs`：订单超时秒数（订单创建后超过此时间未成交则自动撤单）

- **数量控制参数**
  - `bool _by_rate`：是否按照对手盘挂单数的比例下单，true表示按比例（rate字段生效），false表示固定数量（lots字段生效）
  - `double _order_lots`：单次发单手数（当by_rate为false时使用）
  - `double _qty_rate`：下单手数比例（当by_rate为true时使用，如0.1表示每次下对手盘挂单量的10%）
  - `double _min_open_lots`：最小开仓数量（开仓时，如果差量小于此值则不执行）

- **时间控制参数**
  - `uint32_t _entrust_span`：发单时间间隔（单位：毫秒，两次下单之间的最小时间间隔）

- **临时变量**
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基本属性

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 
 * 返回创建该执行单元的工厂名称。
 * 
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char* WtMinImpactExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 
 * 返回执行单元的名称，用于标识和管理。
 * 
 * @return const char* 返回执行单元名称字符串（"WtMinImpactExeUnit"）
 */
const char* WtMinImpactExeUnit::getName()
{
	return "WtMinImpactExeUnit";
}
```

#### 初始化执行单元 init

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
这是 **订单状态回报** 的入口。当发出的订单发生状态变化（如部分成交、全部成交、已撤销）时被调用。其核心作用是 **维护本地订单生命周期** 以及 **触发撤单后的重算逻辑**。
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
* **追单计数重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格策略有效，无需再激进追单。
* **触发重算（核心逻辑）**：
  * 如果订单 **已撤销** (`isCanceled`)：
    * 增加 `_cancel_times` 计数。这会影响下一次计算时的挂单价格（在自动价格模式下，撤单次数越多，价格越激进）。
    * 立即调用 `do_calc()`。这与 TWAP 不同，最小冲击单元不等待时间窗口，而是依靠事件驱动，撤单后立即尝试重新计算并补单。
```cpp
/**
 * @brief 订单回报处理实现
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtMinImpactExeUnit::on_order(uint32_t localid, const char* stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是 **行情驱动** 的入口。当收到新的 Tick 数据时被调用。其核心作用是 **更新市场状态**、**检查订单超时** 以及 **驱动核心交易逻辑**。
* **行情更新与过滤**：
  * 校验合约代码是否与 `_code` 一致。
  * 如果是首个 Tick，检查是否在交易时间段内（过滤集合竞价等非连续交易时段）。
  * 更新内部的 `_last_tick` 指针（引用计数管理）。
* **超时检查**：
  * 如果配置了超时时间 (`_expire_secs != 0`) 且当前有未完成订单 (`_orders_mon.has_order()`) 且没有正在进行的撤单 (`_cancel_cnt == 0`)：
    * 遍历检查所有订单，如果订单存活时间超过阈值，则发起撤单。
    * 撤单成功则增加 `_cancel_cnt`。
* **驱动核心逻辑**：
  * 无论是否有操作，只要有新行情且通过过滤，最后都会调用 `do_calc()`。这保证了策略能根据最新价格动态调整（例如价格满足条件时发单）。
```cpp
/**
 * @brief Tick数据回调实现
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtMinImpactExeUnit::on_tick(WTSTickData* newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新持仓和未完成订单数量。
 * 注意：该函数不触发重新计算，因为成交回报会在on_tick中触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtMinImpactExeUnit::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price)
{
	// 不用触发重新计算，这里在ontick里触发（成交回报会在on_tick中触发重新计算）
}
```

#### 下单结果回报处理 on_entrust
处理**发单请求的即时反馈**（同步或准同步回报）。主要用于处理**发单失败**（如拒单、废单）的情况，确保策略状态能及时回滚或重试。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
    * **立即触发重算**：调用 `do_calc()`。
    * *目的：发单失败意味着本该发出去的量没发出去，需要立即重新评估并再次尝试下单，而不是等待下一个时间窗口
```cpp
/**
 * @brief 下单结果回报处理实现
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtMinImpactExeUnit::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的目标仓位 set_position
这是 **外部控制接口**。通常由上层策略调用，用于设置该合约的目标仓位。
* **清仓保护逻辑**：
  * 如果当前状态已经是“清仓中” (`is_clear(_target_pos)`)，且传入的新目标仓位是 0：返回
  * 这是为了防止清仓过程中被外部逻辑意外重置中断。
* **变动检查**：
  * 如果新目标 `newVol` 与当前目标 `_target_pos` 相等，直接返回，不进行多余计算。
* **更新状态**：
  * 更新 `_target_pos` 为新值 newVol
* **立即触发**：
  * 调用 `do_calc()`，立即根据新的目标仓位开始执行交易逻辑。
```cpp
/**
 * @brief 设置新的目标仓位实现
 * @param stdCode 合约代码
 * @param newVol 新的目标仓位（正数表示多头，负数表示空头，DBL_MAX表示清仓）
 */
void WtMinImpactExeUnit::set_position(const char* stdCode, double newVol)
```

#### 清理全部持仓 clear_all_position
```cpp
/**
 * @brief 清理全部持仓实现
 * 清理指定合约的全部持仓，将所有订单撤单并清空目标仓位。
 * @param stdCode 合约代码
 */
void WtMinImpactExeUnit::clear_all_position(const char* stdCode)
{
	if (_code.compare(stdCode) != 0)  // 如果合约代码不匹配
		return;
	_target_pos = DBL_MAX; // 设置目标仓位为DBL_MAX（清仓标志）
	do_calc(); // 触发执行计算，开始清仓
}
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
这是 **系统初始化/恢复** 的入口。当交易通道连接成功（如程序启动或断线重连）时调用。其核心作用是 **同步本地状态与柜台状态，清理异常订单**。
* **获取柜台未完成单** (`undone`)。
* **场景一：有未完成单，但本地无记录** (`undone != 0 && !_orders_mon.has_order()`)：
  * 说明这些单子是上次运行遗留的或外部下的单，不在本单元监控范围内。
  * **动作**：
    * 撤销 `_ctx->cancel`
    * 并将这些撤单纳入 `_orders_mon` 以及 `_cancel_cnt` 来监控
* **场景二：无未完成单，但本地有记录** (`undone == 0 && _orders_mon.has_order()`)：
  * 说明发生了“错单”现象，通常是断线重连后，本地以为有单，但实际上柜台没收到或已成交/撤销。
  * **动作**：清空本地订单记录 (`_orders_mon.clear_orders()`)，避免逻辑死锁。
* **触发执行**：
  * 状态同步完成后，调用 `do_calc()` 开始正常的交易逻辑。
```cpp
/**
 * @brief 交易通道就绪回调实现
 */
void WtMinImpactExeUnit::on_channel_ready()
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtMinImpactExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
最小冲击执行单元的**核心决策大脑**。它负责将宏观的目标仓位转化为微观的每一笔下单指令。
- **并发与状态前置检查**
  * **并发锁 (`CalcFlag`)**：
    * **作用**：防止 `on_tick`（行情驱动）和 `set_position`（策略驱动）在多线程环境下同时调用 `do_calc`，避免状态错乱。如果检测到正在计算，直接返回。
  * **在途撤单检查 (`_cancel_cnt`)**：
    * 检查 `_cancel_cnt != 0`。
    * **作用**：如果有正在发往柜台的撤单请求尚未收到回报，系统处于不稳定中间态，此时绝对不能发起新的计算或下单，必须等待撤单完成。
  * **线程互斥锁 (`_mtx_calc`)**：
    * 获取 `StdUniqueLock`。
    * **作用**：即使通过了原子标志检查，在读取和修改关键成员变量（如目标仓位）时仍需互斥锁，确保数据一致性。源码注释提到这是为了解决实盘中 `set_position` 和 `on_tick` 并发导致的问题。
- **仓位差额计算与反向单处理**
  * **获取核心数据**：
    * `newVol`: 真实目标仓位 `get_real_target(_target_pos)`
    * `undone`: 当前未完成（挂单）数量 `_ctx->getUndoneQty(stdCode)`
    * `realPos`: 当前实际持仓 `_ctx->getPosition(stdCode)`
    * `diffPos`: **目标差量** = `newVol - realPos`。
  * **反向挂单清洗**：
    * 判断 `diffPos * undone < 0`。
    * **场景**：目标是买入（差量为正），但当前却挂着卖单（未完成为负）；或者目标是卖出，挂着买单。这通常发生在目标仓位突然反转时。
    * **动作**：立即调用 `_ctx->cancel` 撤销所有反向订单，增加 `_cancel_cnt`，并**立即返回**，等待撤单回调。
- **逐笔发单控制**
  * **同向挂单检查**：
    * 判断 `undone != 0`（代码运行至此已排除了反向单，所以这里必然是同向单）。
    * **逻辑**：如果当前还有未成交的同向订单（例如上一笔拆单还没成交），则**返回**。
- **信号过滤与时效性检查**
  * **行情检查**：如果没有 `_last_tick`，无法计算价格，记录日志并**返回**
  * **发单间隔流控 (`_entrust_span`)**：
    * 计算当前时间 `now` 与 `_last_place_time` 的差值。
    * **逻辑**：如果小于配置的 `span`（毫秒），**返回**。防止高频发单被柜台拒单或造成不必要的刷单。
- **完成判断与清仓特判**
  * **目标达成检查**：
    * 如果 `_ctx->getPosition(stdCode) == get_real_target(_target_pos)`
    * **非清仓**（`_target_pos` 不是DBL_MAX）：任务完成，直接返回。
    * **清仓**
      * 如果多头持仓 `_ctx->getPosition(code, true, 1)` 为 0
        * 返回
      * 否则（由于精度问题残留的 *碎股*）
        * 须设立目标仓位 `newVol = -min(_ctx->getPosition(code, true, 1), _order_lots)` （反向发单进行清理）
- **行情更新检查**：防止同一笔行情数据触发两次下单
  * **时间戳比对**：
    * 比较当前 Tick 时间与 `_last_tick_time`。
    * **逻辑**：如果 Tick 时间没有更新（旧数据），直接返回。这常用于防止逻辑被多次调用时重复利用同一瞬间的价格下单。
- **下单数量计算**：计算这一笔子单具体下多少手 `this_qty`
  * **基础数量策略**：
    * **固定手数模式**：默认使用配置的 `_order_lots`。
    * **比例模式 (`_by_rate`)**：
      * 如果是买，取卖一量 (`askqty`)；如果是卖，取买一量 (`bidqty`)。
      * `this_qty = round(this_qty * _qty_rate)`
      * 保底至少下 1 手。
  * **修正一：总差量封顶**：
    * `this_qty = min(this_qty, abs(newVol - curPos))`。
    * 防止最后收尾时，差量只剩 1 手，却按默认的 10 手下单，导致仓位过头。
  * **修正二：平仓封顶**：
    * 判断是否为平仓（`isOpen == false`）。
    * 如果是平仓，`this_qty = min(this_qty, abs(curPos))`。
    * 防止在净仓位模式下，平仓单量超过持仓量导致变成反向开仓（虽然总差量封顶已涵盖此逻辑，但这层是双重保险，特别是针对今昨仓复杂的场景）。
  * **修正三：最小开仓限制**：
    * 如果是开仓 (`isOpen == true`) 且数量小于配置的 `_min_open_lots`。
    * 强制提升至 `_min_open_lots`。这解决了有些策略算出来只开 1 手，但为了节省手续费或满足策略门槛，希望凑够一定数量再开的场景。
- **下单价格计算**：根据配置的模式计算委托价格 `buyPx` 和 `sellPx`
  * **模式 2（`_price_mode` == 2）：自动价格**：
    * **盘口压力指标**：计算 `mp = (买一量 - 卖一量) / (买一价 + 卖一价)`。
    * **趋势判断**：
      * `mp > 0`（买盘强）：买入用卖一价（主动吃），卖出用卖一价（被动挂）。
      * `mp < 0`（卖盘强）：买入用买一价（被动挂），卖出用买一价（主动吃）。
      *  **买一价**：买单中的最高价格。**卖一价**：卖单中的最低价格
    * **0价修正**：如果取到的价格是 0（可能是涨跌停或数据缺失），回退使用最新价或昨收价。
    * **追单逻辑（核心）**：
      * `buyPx += PriceTick * _cancel_times`
      * `sellPx -= PriceTick * _cancel_times`
      * **逻辑**：利用 `_cancel_times`（撤单次数）作为激进因子。如果上一笔单超时没成交被撤了，说明价格不够有吸引力，这次就加价（买）或降价（卖）。撤单次数越多，价格越激进。
  * **其他模式**：
    * **最优价 (`_price_mode` == -1)**：买入用买一，卖出用卖一。
    * **最新价 (`_price_mode` == 0)**：买卖都用最新价。
    * **对手价 (`_price_mode` == 1)**：买入用卖一，卖出用买一。
    * **价格偏移**：在上述基准上，应用 `_price_offset`（如买入+1跳）。
- **涨跌停修正与风控**：确保价格不越界，并设置订单属性。
  * **涨停修正**：如果 `buyPx > _last_tick.upper_limit`，修正为涨停价。
  * **跌停修正**：如果 `sellPx < _last_tick.lower_limit`，修正为跌停价。
  * **不可撤单标记 (`isCanCancel`)**：
    * 如果价格被修正到了涨跌停板，将 `isCanCancel` 设为 `false`。
    * **原因**：在涨跌停板上的挂单通常不需要超时撤单，因为这是能报出的极限价格，撤了再报也一样，不如排队。
- **执行下单**
  * 根据 `isBuy` 方向调用 `_ctx->buy` 或 `_ctx->sell`。
  * 传入计算好的 `buyPx/sellPx`、修正后的 `this_qty` 以及 `bForceClose`（是否强平标记）。
  * 将返回的订单 ID 列表推入 `_orders_mon` 进行监控，同时记录当前时间 `now` 和 `isCanCancel` 标记。
  * 更新 `_last_place_time` 为当前时间。

```cpp
/**
 * @brief 执行计算实现
 */
void WtMinImpactExeUnit::do_calc()
```

## 股票最小冲击执行单元类 WtStockMinImpactExeUnit.h/cpp
```cpp
class WtStockMinImpactExeUnit : public ExecuteUnit
```

### 设计思想
专门为**股票和可转债**市场设计的算法交易策略。它的核心目标和期货版本一致：**隐藏大资金的真实意图，防止因为一笔大单导致股价剧烈波动（滑点）**。

但由于股票市场有 **T+1 交易制度**、**100股为一手（或科创板200股）** 等特殊规则，它的“最小冲击”实现比期货版本更细腻。以下结合代码实际逻辑，为您清晰解析：

1. **核心策略：冰山拆单（化整为零）**
   * **实际场景：** 假设你要买入 **10,000股** 茅台。如果直接发一个市价单，可能会把卖一到卖五的挂单全部扫光，瞬间把价格拉高几个点，成本剧增。
   * **代码体现（逐笔控制）：**
     * 系统每次进入计算（`do_calc`），第一件事就是看“手里有没有还没成交的单子”（`undone`）。
     * 只要**有**任何未完成的挂单，系统就**绝对不发新单**。必须等这一小笔单子成交或撤销后，才进行下一轮。
     * **效果：** 市场里永远只看得到你挂的一笔小单（例如 200 股），像冰山一角，主力资金的意图被完美隐藏。
2. **数量控制：入乡随俗（整手与碎股）**
   * **实际场景：** 股票买卖必须是 100 股的整数倍（买入），卖出时不足 100 股的零头（碎股）必须一次性卖掉。如果算法乱报单（比如报 150 股买单），会被交易所直接拒单，或者暴露算法特征。
   * **代码体现（动态取整）：**
     * **随行就市：** 开启 `_by_rate`（比例模式）后，系统会看对手盘有多少量。比如卖一挂了 1000 股，设定 10% 比例，系统就算出要买 100 股。
     * **强制取整：** 算出来的数量，会强制按 `_min_order` 取整。
       * 普通股票：按 **100股** 取整。
       * 科创板：按 **200股** 取整。
       * 可转债：按 **10张** 取整。
     * **碎股处理（精细化）：** 如果是卖出且剩余持仓不足一手（比如剩 50 股），算法会自动检测到，并生成一笔 **50股** 的卖单一次性卖出，而不是死板地报错。
3. **价格策略：被动吸筹（不吃流动性）**
   * **实际场景：** 你想买入，如果直接挂涨停价去“扫货”，就是主动吃流动性，冲击最大。最小冲击策略倾向于挂在买一价“守株待兔”。
   * **代码体现（智能定价）：**
     * **自动价格（AUTOPX）：** 算法会计算 **买卖压力 (mp)**。
       * 如果卖盘压力大（大家都在卖），算法就挂在 **买一价** 等着别人砸给你（被动成交，冲击最小）。
       * 如果买盘太强，怕买不到，算法才会挂 **卖一价** 去吃单。
     * **温和追单：** 如果挂单挂了 3 秒没成交（超时撤单），说明价格太低。下一次挂单时，算法会自动加一个价位（`PriceTick`）。这种“试探 -> 撤单 -> 加价”的循环，避免了上来就报高价。
     * **引用：** `buyPx += _comm_info->getPriceTick() * _cancel_times;`
4. **节奏控制：避免高频刷单**
   * **实际场景：** 如果一笔单子刚成交，下一笔单子 1 毫秒后马上跟进，很容易被市场中的高频机器人识别并针对（Front-running）。
   * **代码体现（强制冷却）：**
     * 每次下单后，系统会记录时间 `_last_place_time`。
     * 在计算下一笔单子前，检查当前时间。如果距离上次下单不足设定间隔（`_entrust_span`，例如 500毫秒），强制休息，不发单。
     * **效果：** 将交易分散在时间轴上，降低瞬时冲击。
     * **引用：** `if (_now - _last_place_time < _entrust_span) return;`
5. **资金与持仓校验：T+1 的特殊风控**
   * **实际场景：** 股票今天买了不能卖（T+1）。如果算法不仅没隐藏好，还因为计算错误导致频繁废单（想卖但没可用持仓），会引起交易所风控关注。
   * **代码体现（刚性约束）：**
     * **卖出限制：** 每次计算时，严格使用 `vailyPos`（可用持仓，即昨仓）来限制卖出量。
       * 算法会自动识别 **T+0** 品种（如可转债），允许卖出今仓。
       * 对于股票，通过 `target_pos = max(curPos - vailyPos, _target_pos)` 锁死今仓，防止误操作。
     * **买入限制：** 实时计算 `_avaliable`（可用资金），确保拆出来的每一笔小单都能成交，不会因为资金不足产生废单冲击。
     * **引用：** `double vailyPos = _ctx->getPosition(stdCode, true);`
**总结：一个生动的股票买入例子**
* 假设你要买入 **5000股** 某科创板股票（最小200股）：
  1. **启动：** 算法发现目标差额 5000股。
  2. **侦查：** 发现卖一盘口只有 1000股。
  3. **计算：** 设定不吃光盘口，只吃 20%。计算出 `1000 * 20% = 200股`。
  4. **下单：** 挂单买入 **200股**，价格挂在买一（排队等待，不推高股价）。
  5. **循环：**
     * 如果成交了，**休息 500毫秒**，然后检查还剩 4800股，继续下一轮。
     * 如果没成交（超时），撤单，下次提价一档重新挂。
  6. **收尾：** 最后剩 50股买不到了（不足200股），或者资金不够买200股了，算法会自动停止或调整，不会死循环报错。

通过这种 **“切蛋糕（拆单） -> 看盘口（定合规手数） -> 挂单排队（定价） -> 休息（流控）”** 的流程，`WtStockMinImpactExeUnit` 实现了在复杂的股票规则下，悄无声息地完成大额交易。

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _target_pos`：目标仓位（股数模式下的目标股数，正数表示买入，负数表示卖出，DBL_MAX表示清仓）
  - `double _target_amount`：目标金额（金额模式下的目标金额，正数表示买入金额，负数表示卖出金额）
  - `double _target_ratio`：目标持仓比例（比例模式下的目标比例，0-1之间的值）
  - `double _avaliable`：账户可用资金（用于金额模式和比例模式的计算）

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **价格控制参数**
  - `int32_t _price_offset`：价格偏移跳数（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `int32_t _price_mode`：价格模式：0-最新价，-1-最优价，1-对手价，2-自动价格
  - `uint32_t _expire_secs`：订单超时秒数（订单创建后超过此时间未成交则自动撤单）

- **数量控制参数**
  - `bool _by_rate`：是否按照对手盘挂单数的比例下单，true表示按比例（rate字段生效），false表示固定数量（lots字段生效）
  - `double _order_lots`：单次发单手数（当by_rate为false时使用）
  - `double _qty_rate`：下单手数比例（当by_rate为true时使用，如0.1表示每次下对手盘挂单量的10%）
  - `double _min_open_lots`：最小开仓数量（开仓时，如果差量小于此值则不执行）

- **时间控制参数**
  - `uint32_t _entrust_span`：发单时间间隔（单位：毫秒，两次下单之间的最小时间间隔）

- **股票市场特殊参数**
  - `TargetMode _target_mode`：目标模式（股数/金额/比例），枚举值：stocks=0（股数模式），amount=1（金额模式），ratio=2（比例模式）
  - `bool _is_KC`：是否是科创板股票标志（科创板代码>=688000）
  - `double _min_hands`：最小手数（根据股票类型自动计算：普通股票100股，科创板200股，可转债10张）
  - `bool _is_t0`：是否T+0交易标志（对于转债等来说，这个需要是true，股票为false）

- **错单检测参数**
  - `uint32_t _max_cancel_times`：最大撤单次数（超过此次数仍无法撤单则视为错单）
  - `bool _auto_cancel_unmanager`：是否自动撤单未管理订单标志（true表示自动撤单，false表示不撤单）

- **临时变量**
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基本属性

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 
 * 返回创建该执行单元的工厂名称。
 * 
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char* WtStockMinImpactExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 
 * 返回执行单元的名称，用于标识和管理。
 * 
 * @return const char* 返回执行单元名称字符串（"WtStockMinImpactExeUnit"）
 */
const char* WtStockMinImpactExeUnit::getName()
{
	return "WtStockMinImpactExeUnit";  // 返回执行单元名称
}
```

#### 初始化执行单元 init

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
这是 **订单状态回报** 的入口。当发出的订单发生状态变化（如部分成交、全部成交、已撤销）时被调用。其核心作用是 **维护本地订单生命周期** 以及 **触发撤单后的重算逻辑**。
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
* **追单计数重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格策略有效，无需再激进追单。
* **触发重算（核心逻辑）**：
  * 如果订单 **已撤销** (`isCanceled`)：
    * 增加 `_cancel_times` 计数。这会影响下一次计算时的挂单价格（在自动价格模式下，撤单次数越多，价格越激进）。
    * 立即调用 `do_calc()`。这与 TWAP 不同，最小冲击单元不等待时间窗口，而是依靠事件驱动，撤单后立即尝试重新计算并补单。

```cpp
/**
 * @brief 订单回报处理实现
 * 
 * 当订单状态发生变化时调用此函数，更新订单状态，处理撤单等操作。
 * 
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtStockMinImpactExeUnit::on_order(uint32_t localid, const char* stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是 **行情驱动入口**。当收到新的 Tick 数据时被调用，负责更新状态、检查异常并驱动策略计算。
* **更新环境与校验**：
  * 更新本地时间 `_now`。
  * 校验 Tick 数据有效性及合约代码是否匹配。
  * 如果是第一笔 Tick，根据 `_sess_info` 检查是否在交易时段内，过滤集合竞价期间的数据。
* **重复行情过滤**：
  * 比较当前 Tick 时间与 `_last_tick_time`。如果时间戳未更新（收到了重复或乱序的行情），直接返回，防止重复计算。
* **状态维护**：
  * 释放旧 Tick，持有新 Tick 到 `_last_tick`。
  * 调用 `_orders_mon.enumOrder` 打印当前所有在途订单的状态日志。
* **超时检查与撤单**：
  * 如果配置了超时时间 (`_expire_secs != 0`) 且有在途订单（`_orders_mon` 非空）：
    * 遍历检查订单是否超时。
    * 对超时订单调用 `_ctx->cancel` 进行撤单。
    * **撤单计数**：如果撤单请求发送成功，在 `_cancel_map` 中记录该 `localid` 的撤单次数。
* **错单/僵尸单强制清理**：
  * **逻辑**：检查 `_cancel_map`。如果某个订单的撤单次数超过了 `_max_cancel_time`（说明多次尝试撤单都未收到成功的撤单回报，可能卡住了），将其视为错单。
  * **动作**：
    * 强制从 `_orders_mon` 和 `_cancel_map` 中移除该订单
    * *目的：* 防止因为一笔卡住的订单导致策略永远无法进入下一轮下单逻辑。
* **驱动计算**：
  * 最后调用 `do_calc()`，利用最新的行情数据进行下单决策。
```cpp
/**
 * @brief Tick数据回调实现
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtStockMinImpactExeUnit::on_tick(WTSTickData* newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新持仓和未完成订单数量。
 * 注意：该函数不触发重新计算，因为成交回报会在on_tick中触发重新计算。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtStockMinImpactExeUnit::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price)
{
	// 成交回报会在on_tick中触发重新计算（当前版本不做任何处理）
}
```

#### 下单结果回报处理 on_entrust
处理**发单请求的即时反馈**（同步或准同步回报）。主要用于处理**发单失败**（如拒单、废单）的情况，确保策略状态能及时回滚或重试。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
* **立即触发重算**：调用 `do_calc()`。

```cpp
/**
 * @brief 下单结果回报处理实现
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtStockMinImpactExeUnit::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的目标仓位 set_position
这是 **外部控制接口**。由上层策略调用，设置目标持仓（股数模式）。
* **安全检查**：
  * 校验合约代码是否与 `_code` 一致
  * **清仓保护**：如果当前处于清仓模式 (`is_clear()`) 且新目标 newVol 为 0，视为重复指令，直接返回
  * **变动检查**：如果当前仓位 `_ctx->getPosition(stdCode)` 已经等于新目标 newVol，直接返回。
  * **合法性检查**：股票不支持负数目标仓位（做空通常通过融券，但此处逻辑不支持负数作为目标），若 `newVol< 0` 则返回。
* **状态重置**：
  * 更新 `_target_pos` 为新目标 newVol
  * 设置 `_target_mode` 为 `TargetMode::stocks`。
  * 重置完成标志 `_is_finish = false`，激活策略执行。
  * 更新 `_start_time` 为本地时间
* **基准价格记录**：
  * 尝试获取最新 Tick 价格 `_ctx->grabLastTick(_code.c_str()).price()` 更新 `_start_price`
* **立即触发**：
  * 调用 `do_calc()`，立即根据新目标开始执行。
```cpp
/**
 * @brief 设置新的目标仓位实现
 * @param stdCode 合约代码
 * @param newVol 新的目标仓位（正数表示目标股数，0表示清仓，负数表示错误值）
 */
void WtStockMinImpactExeUnit::set_position(const char* stdCode, double newVol)
```

#### 清理全部持仓 clear_all_position
```cpp
/**
 * @brief 清理全部持仓实现
 * 
 * 设置执行单元为清仓模式，将目标仓位设置为0，执行单元会将所有持仓卖出。
 * 
 * @param stdCode 合约代码
 */
void WtStockMinImpactExeUnit::clear_all_position(const char* stdCode)
{
	if (_code.compare(stdCode) != 0)  // 如果合约代码不匹配
		return;  // 直接返回，不做任何处理

	_is_clear = true;  // 设置清仓标志为true
	_target_pos = 0;  // 设置目标仓位为0（清仓）
	_target_amount = 0;  // 设置目标金额为0（清仓）
	do_calc();  // 触发执行计算，开始清仓
}
```

### ExecuteUnit—账户信息回调接口

#### 账户信息回调 on_account
```cpp
/**
 * @brief 账户信息回调实现
 * 
 * 当账户信息更新时调用此函数，更新可用资金等信息。
 * 
 * @param currency 货币类型（如"CNY"表示人民币）
 * @param prebalance 上日余额
 * @param balance 当前余额
 * @param dynbalance 动态余额
 * @param avaliable 可用资金（买入时需要考虑此限制）
 * @param closeprofit 平仓盈亏
 * @param dynprofit 浮动盈亏
 * @param margin 保证金
 * @param fee 手续费
 * @param deposit 入金
 * @param withdraw 出金
 */
void WtStockMinImpactExeUnit::on_account(const char* currency, double prebalance, double balance, double dynbalance, double avaliable, double closeprofit, double dynprofit, double margin, double fee, double deposit, double withdraw)
{
	if (strcmp(currency, "CNY") == 0) // 如果是人民币账户
	{
		_ctx->writeLog(fmtutil::format("avaliable update {}->:{}", _avaliable, avaliable)); // 记录日志：可用资金更新
		_avaliable = avaliable; // 更新可用资金（买入时需要考虑此限制）
	}
}
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
```cpp
/**
 * @brief 交易通道就绪回调实现
 * 
 * 当交易通道连接成功并准备就绪时调用此函数，可以开始下单。
 * 该函数会检查是否有未管理的订单（可能是上次启动时的未完成单或外部挂单），
 * 如果有则自动撤单，然后触发执行计算。
 */
void WtStockMinImpactExeUnit::on_channel_ready()
{
	_ctx->writeLog("=================================channle ready=============================="); // 记录日志：交易通道就绪
	_is_ready = true; // 设置就绪标志为true
	check_unmanager_order(); // 检查并处理未管理的订单
	do_calc(); // 触发执行计算，开始下单
}
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtStockMinImpactExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
这是 **股票执行核心大脑**。处理T+1、资金检查、碎股处理、价格计算和下单执行。

1. **前置防御与并发控制**
   * 在进入复杂的计算之前，先确保环境安全、状态正确。
   * **状态检查**：
     * 如果不持有行情 (`!_last_tick`)、任务已完成 (`_is_finish`) 或通道/资金未就绪 (`!_is_ready`)，直接返回。
   * **线程安全 (`StdUniqueLock`)**：
     * 获取 `_mtx_calc` 互斥锁。
     * **作用**：防止 `on_tick` (行情驱动) 和 `set_position` (策略驱动) 在多线程环境下同时修改关键变量，导致计算错乱。
2. **T+1 规则与可用持仓计算**
   * 这是股票与期货最大的区别点，必须区分“今仓”和“昨仓”。
   * **获取持仓**：
     * `curPos`: 总持仓（昨仓 + 今买）。
     * `vailyPos`: **可用持仓**。
       * 普通股票：等于昨仓（今日买入的不能卖）。
       * T+0 品种（如可转债 `_is_t0`）：等于总持仓（今日买入的可以卖）。
   * **目标仓位修正（核心风控）**：
     * `target_pos = max(curPos - vailyPos, _target_pos)`
     * **逻辑**：理论最小持仓是今仓（T+1）或 0（T+0），目标仓位不能低于这个最小持仓
3. **完成判断与“碎股”陷阱**
   * 判断任务是否结束，需要极高精度的比较，并处理“不到一手”的特殊情况。
   * **差量计算**：`diffPos = target_pos - curPos`。
   * **模糊比较（Round）**：
     * 使用 `round_hands` 将目标和当前仓位按 `_min_hands`（如100股）取整后比较。避免因为计算机浮点数精度问题导致永远无法相等。
   * **碎股（零股）特判**：
     * 条件：`!(target_pos == 0 && curPos < _min_hands && curPos > target_pos)`
     * **场景**：你需要清仓 (`target_pos == 0`)，但账户里还剩 50 股 (`curPos < 100`)。
     * **逻辑**：如果只看取整比较，0 和 50 取整后可能都视为 0，会导致策略提前退出。这个条件强制要求：**如果是清仓且有碎股残余，不能算完成**，必须进入后续逻辑把这 50 股卖掉。
4. **反向单与逐笔控制**
   * 清理干扰订单，并保持“一笔一笔做”的节奏。
   * **反向单清洗**：
     * 如果 `diffPos`（目标方向）与 `undone`（挂单方向）相反，说明行情或目标突变。
     * **动作**：立即撤销反向挂单，返回等待。
   * **同向逐笔控制**：
     * 如果还有同向未完成单 (`undone != 0`)，**直接返回**。
     * **原则**：最小冲击策略不进行批量挂单，必须等上一笔完全结束（成交或撤单）才发下一笔。
5. **流控与时效**
   * **行情时效**：如果没有 Tick 数据，无法定价，返回。
   * **发单间隔 (`_entrust_span`)**：
     * 如果距离上次下单时间太短，强制等待。防止高频刷单。
6. **下单数量计算：取整与资金限制**
   * 计算 `this_qty`，这是最复杂的数学部分，涉及多重约束。
   * **基础量计算**：
     * **固定模式**：`_order_lots`。
     * **比例模式 (`_by_rate`)**：取对手盘口量 * 比例。**关键修正**：必须按 `_min_order`（用户设定或品种限制，如200股）取整，且不能小于最小下单数。
   * **总差量封顶**：`min(this_qty, diffPos)`。
   * **买入逻辑**：
     * **取整**：必须是 `_min_order` 的整数倍（如买入只能是100的倍数）。
     * **资金验算**：
       * `max_can_buy = _avaliable / price`（算出钱够买多少股）。
       * 向下取整到 `_min_order` 整数倍。
       * 取 `min`，确保不透支资金。
   * **卖出逻辑 - 碎股核心**：
     * **碎股检测**：如果 `vailyPos < _min_order`（例如可用50股，最小下单100股）。
       * **动作**：`this_qty = vailyPos`。允许直接卖出 50 股（交易所通常允许一次性卖出不足一手的零股）。
     * **正常卖出**：如果可用大于一手，则按 `_min_order` 取整。
     * **可用封顶**：`min(vailyPos, this_qty)`，确保不卖空（T+1限制）。
7. **价格计算与追单**
   * **自动价格 (AUTOPX)**：
     * 计算多空压力 `mp`。多头强则主动吃（卖一价），空头强则被动挂（买一价）。
   * **其他模式**：最优价、最新价、对手价 + 偏移。
   * **0价保护**：如果行情缺失导致价格为0，回退使用昨收价。
   * **追单机制 (`_cancel_times`)**：
     * `Price += Tick * _cancel_times` (买)
     * `Price -= Tick * _cancel_times` (卖)
     * **逻辑**：如果上一笔单子超时没成交被撤了，说明价格不够优，这次根据撤单次数加价/降价，次数越多越激进。
8. **涨跌停风控**
   * **修正**：如果买价 > 涨停，修正在涨停价；卖价 < 跌停，修正在跌停价。
   * **标记不可撤 (`isCanCancel = false`)**：
     * 涨跌停板上的单子通常不进行超时撤单，因为这是你能报的最优价，撤了再报还得重新排队，不如死等。
9. **执行下单**
   * 调用 `_ctx->buy` 或 `_ctx->sell`。
   * 将返回的订单 ID 加入 `_orders_mon` 进行全生命周期监控。
   * 更新 `_last_place_time`。

```cpp
/**
 * @brief 执行计算实现
 */
void WtStockMinImpactExeUnit::do_calc()
```

#### 检查是否清仓 is_clear
```cpp
/**
 * @brief 检查是否清仓实现
 * 
 * 检查执行单元是否处于清仓模式。
 * 
 * @return bool true表示正在清仓，false表示不清仓
 */
inline bool WtStockMinImpactExeUnit::is_clear()
{
	return _is_clear;  // 返回清仓标志
}
```

#### 检查未管理订单 check_unmanager_order
这是 **环境清理逻辑**。通常在 `on_channel_ready`（通道就绪）时调用，用于处理策略启动前或外部遗留的挂单。
* **获取柜台状态**：
  * 调用 `_ctx->getUndoneQty` 获取当前柜台上该合约的所有未完成挂单总量。
* **重置本地监控**：
  * 调用 `_orders_mon.clear_orders()`，清空本地记录，准备重新接管。
* **自动撤单逻辑**：
  * 如果发现有未完成单 (`undone != 0`) **且** 配置允许撤销 (`_is_cancel_unmanaged_order` 为 true)：
  * **动作**：
    * 调用 `_ctx->cancel` 撤销这些遗留订单。
    * **接管撤单状态**：将撤单返回的 `OrderIDs` 推入 `_orders_mon`。
    * *目的：* 确保策略在干净的环境下运行，同时监控这些“清洗动作”产生的撤单请求，直到它们真正完成。
```cpp
/**
 * @brief 检查未管理订单实现
 */
void WtStockMinImpactExeUnit::check_unmanager_order()
```

## 差量最小冲击执行单元 WtDiffMinImpactExeUnit.h/cpp
```cpp
class WtDiffMinImpactExeUnit : public ExecuteUnit
```

### 设计思想
专门为**增量交易**设计的执行策略。与标准版不同，它不关心 *现在的仓位是多少* 或 *最终要到多少仓位*，它只关心 *我现在需要立刻买入/卖出多少手*。通常用于**高频策略、算法拆单或手动干预**，即明确指令是“再买500手”，而不是“把仓位调整到500手”。
1. **核心逻辑：只盯着“剩余差量”**
   * **实际场景：** 假设你突然收到指令要“立刻买入 1000 手”。如果直接扔进市场，会瞬间吃光卖盘。此单元的做法是维护一个计数器 `_left_diff`（剩余差量）。
   * **代码体现：**
     * **设置任务：** 当外部调用 `set_position` 时，传入的数值被直接作为“待执行任务量”存入 `_left_diff`（例如 +1000）。
     * **递减任务：** 每当有一笔子单成交（`on_trade`），系统会自动从 `_left_diff` 中减去成交量。
     * **任务结束：** 当 `_left_diff` 归零时，算法停止。
     * **引用：** `_left_diff -= vol * (isBuy ? 1 : -1);`
2. **隐身术：逐笔拆单**
   * **实际场景：** 就像搬运 1000 块砖头，不是一次搬完，而是一次搬 10 块，搬完再回来搬下一趟。
   * **代码体现：**
     * 在 `do_calc` 中，系统会检查 `undone`（当前未成交挂单）。
     * 只要有**任何**未完成的挂单（哪怕只剩 1 手在排队），系统就**拒绝发出新指令**。
     * **效果：** 市场永远只能看到你的一笔小单。前一笔不结束（成交或撤单），后一笔绝不进场。
     * **引用：** `if (!decimal::eq(undone, 0)) ... return;`
3. **数量控制：精确封顶**
   * **实际场景：** 随着执行的进行，还剩最后 5 手没买。如果这时候按照默认逻辑下 10 手，就买多了（Over-fill）。
   * **代码体现：**
     * **封顶逻辑：** 每次计算下单量 `this_qty` 时，都会和 `abs(diffPos)`（剩余差量）取较小值。
     * **平仓保护：** 如果是在平仓（比如 `_left_diff` 是卖出，而当前持有多单），还会限制下单量不超过当前持仓 `curPos`，防止因为数据延迟导致“平仓变成了反手开空”。
     * **引用：** `this_qty = min(this_qty, abs(diffPos));`
4. **价格策略：试探与追单**
   * **实际场景：** 为了不推高价格，一开始挂单很保守（挂买一）。如果没人卖给你，说明价格低了，需要一点点加价。
   * **代码体现：**
     * **自动定价：** 根据盘口压力自动选择是“被动挂单”还是“主动吃单”。
     * **超时撤单与追单：** 订单发出后 `_expire_secs` 秒不成交，撤单。撤单后，`_cancel_times` 计数增加。
     * **动态加价：** 下一次挂单价格 = `基准价 + (最小跳动 * 撤单次数)`。撤得越多，价格越激进，直到成交为止。
     * **引用：** `buyPx += _comm_info->getPriceTick() * _cancel_times;`
5. **节奏控制：时间冷却**
   * **实际场景：** 机器人如果连续 1 秒内发几十个单子，会被交易所风控。
   * **代码体现：**
     * 每次发单后记录时间。下一笔计算时，如果距离上次不足 `_entrust_span`（如 500ms），强制等待。
     * **引用：** `if (now - _last_place_time < _entrust_span) return;`


**一个差量执行例子**
* 假设你通过算法发出指令：**“做多 500 手”**。
  1. **接令：** `_left_diff` 变为 +500。
  2. **拆解：** 设定每次只买 10 手。
  3. **第一笔：** 挂买入 10 手，价格放在买一价。
  4. **等待：** 
     * 如果成交了：`_left_diff` 变为 490。休息 500ms，准备下一笔。
     * 如果没成交（超时）：撤单，价格加一跳，重新挂这 10 手。
  5. **循环：** 如此往复 49 次。
  6. **收尾：** 当 `_left_diff` 变成 0 时，任务结束，停止任何动作。

这种机制完美适用于 **"Sniper"（狙击）** 类策略或者 **手动大单拆分**，它不关心账户里原来有多少货，只负责忠实、隐蔽地执行你给它的“增量任务”。

### 成员
- **行情与状态数据**
  - `WTSTickData* _last_tick`：上一笔行情数据指针，用于获取最新价格等信息
  - `double _left_diff`：未执行差量（剩余需要执行的差量，正数表示买入，负数表示卖出）

- **合约与交易信息**
  - `WTSCommodityInfo* _comm_info`：合约信息指针，包含合约的基本信息（如最小变动价位、合约乘数等）
  - `WTSSessionInfo* _sess_info`：交易时段信息指针，包含交易时间、休市时间等信息

- **线程安全与同步**
  - `StdUniqueMutex _mtx_calc`：计算逻辑的互斥锁，保证线程安全
  - `std::atomic<bool> _in_calc`：计算中标志（原子布尔标志，用于防止并发计算）

- **订单管理**
  - `WtOrdMon _orders_mon`：订单管理器，用于跟踪和管理订单状态
  - `uint32_t _cancel_cnt`：在途撤单量（正在撤单的订单数量）
  - `uint32_t _cancel_times`：撤单次数（累计撤单次数统计）

- **价格控制参数**
  - `int32_t _price_offset`：价格偏移跳数（相对于基准价格的偏移，买入+偏移，卖出-偏移）
  - `int32_t _price_mode`：价格模式：0-最新价，-1-最优价，1-对手价，2-自动价格
  - `uint32_t _expire_secs`：订单超时秒数（订单创建后超过此时间未成交则自动撤单）

- **数量控制参数**
  - `bool _by_rate`：是否按照对手盘挂单数的比例下单，true表示按比例（rate字段生效），false表示固定数量（lots字段生效）
  - `double _order_lots`：单次发单手数（当by_rate为false时使用）
  - `double _qty_rate`：下单手数比例（当by_rate为true时使用，如0.1表示每次下对手盘挂单量的10%）

- **时间控制参数**
  - `uint32_t _entrust_span`：发单时间间隔（单位：毫秒，两次下单之间的最小时间间隔）

- **临时变量**
  - `uint64_t _last_place_time`：上个下单时间（毫秒时间戳，用于控制发单间隔）
  - `uint64_t _last_tick_time`：上个tick时间（毫秒时间戳，用于判断是否有新行情）

- **辅助类**
  - `typedef struct _CalcFlag CalcFlag`：计算标志辅助类（RAII辅助类，用于自动管理计算标志的生命周期）
    - `bool _result`：构造时标志的原始值（true表示有并发计算）
    - `std::atomic<bool>* _flag`：指向计算标志的指针

### ExecuteUnit—基类接口实现

#### 获取所属执行器工厂名称 getFactName
```cpp
/**
 * @brief 获取所属执行器工厂名称实现
 * 
 * 返回创建该执行单元的工厂名称。
 * 
 * @return const char* 返回工厂名称字符串（"WtExeFact"）
 */
const char* WtDiffMinImpactExeUnit::getFactName()
{
	return FACT_NAME;
}
```

#### 获取执行单元名称 getName
```cpp
/**
 * @brief 获取执行单元名称实现
 * 
 * 返回执行单元的名称，用于标识和管理。
 * 
 * @return const char* 返回执行单元名称字符串（"WtDiffMinImpactExeUnit"）
 */
const char* WtDiffMinImpactExeUnit::getName()
{
	return "WtDiffMinImpactExeUnit";  // 返回执行单元名称
}
```

#### 初始化执行单元 init

### ExecuteUnit—订单与交易回调接口

#### 订单回报处理 on_order
这是 **订单状态回报** 的入口。当发出的订单发生状态变化（如部分成交、全部成交、已撤销）时被调用。其核心作用是 **维护本地订单生命周期** 以及 **触发撤单后的重算逻辑**。
* **有效性检查**：
  * 检查收到的 `localid` 是否存在于本地订单管理器 `_orders_mon` 中。如果不存在（非本策略订单或已处理），直接返回。
* **状态清理（终态处理）**：
  * 如果订单 **已撤销** (`isCanceled`) 或 **全部成交** (`leftover == 0`)：
    * 从 `_orders_mon` 中移除该订单记录。
    * 如果当前有正在进行的撤单计数 (`_cancel_cnt > 0`)，则递减计数（表示一笔撤单请求已确认完成）。
* **追单计数重置**：
  * 如果订单 **全部成交** 且 **未撤销**：
    * 将 `_cancel_times`（撤单/追单次数计数器）重置为 0。这意味着之前的价格策略有效，无需再激进追单。
* **触发重算（核心逻辑）**：
  * 如果订单 **已撤销** (`isCanceled`)：
    * 增加 `_cancel_times` 计数。这会影响下一次计算时的挂单价格（在自动价格模式下，撤单次数越多，价格越激进）。
    * 立即调用 `do_calc()`。这与 TWAP 不同，最小冲击单元不等待时间窗口，而是依靠事件驱动，撤单后立即尝试重新计算并补单。

```cpp
/**
 * @brief 订单回报处理实现
 * 
 * 当订单状态发生变化时调用此函数，更新订单状态，处理撤单等操作。
 * 
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param leftover 剩余未成交数量
 * @param price 委托价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 */
void WtDiffMinImpactExeUnit::on_order(uint32_t localid, const char* stdCode, bool isBuy, double leftover, double price, bool isCanceled)
```

#### Tick数据回调 on_tick
这是 **行情驱动** 的入口。当收到新的 Tick 数据时被调用。其核心作用是 **更新市场状态**、**检查订单超时** 以及 **驱动核心交易逻辑**。
* **行情更新与过滤**：
  * 校验合约代码是否与 `_code` 一致。
  * 如果是首个 Tick，检查是否在交易时间段内（过滤集合竞价等非连续交易时段）。
  * 更新内部的 `_last_tick` 指针（引用计数管理）。
* **超时检查**：
  * 如果配置了超时时间 (`_expire_secs != 0`) 且当前有未完成订单 (`_orders_mon.has_order()`) 且没有正在进行的撤单 (`_cancel_cnt == 0`)：
    * 遍历检查所有订单，如果订单存活时间超过阈值，则发起撤单。
    * 撤单成功则增加 `_cancel_cnt`。
* **驱动核心逻辑**：
  * 无论是否有操作，只要有新行情且通过过滤，最后都会调用 `do_calc()`。这保证了策略能根据最新价格动态调整（例如价格满足条件时发单）。

```cpp
/**
 * @brief Tick数据回调实现
 * @param newTick 最新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void WtDiffMinImpactExeUnit::on_tick(WTSTickData* newTick)
```

#### 成交回报处理 on_trade
```cpp
/**
 * @brief 成交回报处理实现
 * 
 * 当订单有成交回报时调用此函数，更新未执行差量。
 * 在差量模式下，成交会减少未执行差量。
 * 
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量（这里没有正负，通过isBuy确定买入还是卖出）
 * @param price 成交价格
 */
void WtDiffMinImpactExeUnit::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price)
{
	// 如果是本地订单，则更新差量
	if (!_orders_mon.has_order(localid))  // 如果订单管理器中不存在该订单
		return;
	// 更新未执行差量：买入成交减少差量（正数），卖出成交增加差量（负数）
	_left_diff -= vol * (isBuy ? 1 : -1); // 买入时减去成交量，卖出时加上成交量
	_ctx->writeLog(fmtutil::format("Left diff of {} updated to {}", _code.c_str(), _left_diff));
}
```

#### 下单结果回报处理 on_entrust
处理**发单请求的即时反馈**（同步或准同步回报）。主要用于处理**发单失败**（如拒单、废单）的情况，确保策略状态能及时回滚或重试。
* **失败处理** (`!bSuccess`)：
  * 检查订单是否在 `_orders_mon` 监控中。
    * 从 `_orders_mon` 中移除该订单（因为交易所根本没收，所以不算在途订单）。
    * **立即触发重算**：调用 `do_calc()`。
    * 目的：发单失败意味着本该发出去的量没发出去，需要立即重新评估并再次尝试下单，而不是等待下一个时间窗口
```cpp
/**
 * @brief 下单结果回报处理实现
 * @param localid 本地订单ID
 * @param stdCode 合约代码
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 返回消息（如果失败，包含失败原因）
 */
void WtDiffMinImpactExeUnit::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message)
```

### ExecuteUnit—仓位管理接口

#### 设置新的差量 set_position
```cpp
/**
 * @brief 设置新的差量实现
 * 
 * 设置执行单元的差量指令，执行单元会立即执行该差量。
 * 
 * @param stdCode 合约代码
 * @param newDiff 新的差量指令（正数表示买入N手，负数表示卖出N手）
 * 
 * 注意：差量执行单元不支持清仓指令（DBL_MAX），如果传入DBL_MAX则直接返回。
 */
void WtDiffMinImpactExeUnit::set_position(const char* stdCode, double newDiff)
{
	if (_code.compare(stdCode) != 0)  // 如果合约代码不匹配
		return;
	if (newDiff == DBL_MAX) // 如果差量指令为DBL_MAX（清仓标志）
	{
		_ctx->writeLog("Diff execute unit do not support clear command");  // 记录日志：差量执行单元不支持清仓指令
		return;
	}
	if(_left_diff != newDiff) // 如果差量指令有变化
	{
		_left_diff = newDiff; // 更新未执行差量
		_ctx->writeLog(fmtutil::format("Diff of {} updated to {}", stdCode, _left_diff));
		do_calc(); // 触发执行计算，根据新差量执行
	}
}
```

#### 清理全部持仓 clear_all_position
```cpp
/**
 * @brief 清理全部持仓实现
 * 
 * 差量执行单元不支持清仓指令，该函数直接返回并记录日志。
 * 
 * @param stdCode 合约代码
 */
void WtDiffMinImpactExeUnit::clear_all_position(const char* stdCode)
{
	_ctx->writeLog("Diff execute unit do not support clear command");
	return; // 直接返回，不做任何处理
}
```

### ExecuteUnit—通道状态回调接口

#### 交易通道就绪回调 on_channel_ready
这是 **系统初始化/恢复** 的入口。当交易通道连接成功（如程序启动或断线重连）时调用。其核心作用是 **同步本地与柜台的状态，清理“僵尸单”，并启动策略**。
* **获取柜台未完成单**：
  * 调用 `_ctx->getUndoneQty` 获取当前合约在柜台（交易所/Broker）上的实际未成交挂单总量 (`undone`)。
* **场景一：有未完成单，但本地无记录** (`undone != 0 && !_orders_mon.has_order()`)：
  * **含义**：说明这些挂单是上次运行遗留的、或者是手动下的单，不在本执行单元的监控范围内（无本地 ID）。
  * **动作**：
    * 调用 `_ctx->cancel` **全部撤销**这些订单。
    * 将撤单操作产生的订单 ID 推入 `_orders_mon` 监控，并增加 `_cancel_cnt`（在途撤单计数），确保策略在干净的环境下开始。
* **场景二：无未完成单，但本地有记录** (`undone == 0 && _orders_mon.has_order()`)：
  * **含义**：说明发生了状态不一致（错单）。本地以为有挂单，但柜台实际上没有（可能在断线期间已成交或已撤销）。
  * **动作**：调用 `_orders_mon.clear_orders()` **清空本地订单记录**。这相当于强制同步状态，避免策略死锁在“等待一个不存在的订单”的状态中。
* **触发执行**：
  * 无论上述哪种情况，处理完毕后都会调用 `do_calc()`，正式开始根据当前的 `_left_diff` 进行交易计算。
```cpp
/**
 * @brief 交易通道就绪回调实现
 *
 * 当交易通道连接成功并准备就绪时调用此函数，可以开始下单。
 */
void WtDiffMinImpactExeUnit::on_channel_ready()
```

#### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * 
 * 当交易通道断开时调用此函数，停止下单操作。
 * 当前版本该函数为空，不做任何处理。
 */
void WtDiffMinImpactExeUnit::on_channel_lost()
{
	// 交易通道丢失时，停止下单操作（当前版本不做任何处理）
}
```

### 私有方法

#### 执行计算 do_calc
差量最小冲击执行单元的**核心大脑**。与标准模式不同，它不关心总仓位，而是专注于“还有多少差量没有执行完”。

1. **前置防御与并发控制**
   * 在进入核心计算前，首先确保环境安全，防止多线程冲突。
   * **原子标志锁 (`CalcFlag`)**：
     * 使用 `std::atomic<bool> _in_calc` 防止重入。
     * **作用**：如果 `on_tick`（行情驱动）和 `set_position`（策略驱动）同时触发，保证同一时间只有一个线程在运行计算逻辑。
   * **在途撤单检查 (`_cancel_cnt`)**：
     * 检查 `_cancel_cnt != 0`。
     * **作用**：如果有正在发往柜台的撤单请求尚未收到回报，系统处于不稳定中间态，**直接返回**，等待撤单完成。
   * **线程互斥锁 (`_mtx_calc`)**：
     * 获取 `StdUniqueLock`。
     * **作用**：保护后续对成员变量（如 `_left_diff`）的读取和修改，确保数据一致性。
2. **状态获取与逐笔控制**
   * 获取当前的任务状态，并执行严格的“逐笔”逻辑。
   * **获取核心数据**：
     * `undone`: 当前未完成（挂单）数量 `_ctx->getUndoneQty(stdCode)`。
     * `diffPos`: **未执行差量** (`_left_diff`)。这是本策略的核心，正数代表还需买入多少，负数代表还需卖出多少。
   * **逐笔发单控制（关键）**：
     * 判断 `!decimal::eq(undone, 0)`。
     * **逻辑**：只要当前还有未成交的订单（哪怕只有 1 手），**直接返回**，暂不发单。
     * **目的**：最小冲击策略必须等上一笔子单完全结束（成交或撤单）后，才允许计算下一笔，严禁并发挂单。
   * **任务完成检查**：
     * 如果 `diffPos == 0`，说明任务已完成，直接返回。
3. **信号过滤与时效性检查**
   * **行情检查**：如果没有 `_last_tick`，无法定价，记录日志并返回。
   * **发单间隔流控 (`_entrust_span`)**：
     * 检查 `now - _last_place_time`。如果小于设定间隔，**返回**。防止高频刷单。
   * **买卖方向判定**：
     * `isBuy = diffPos > 0`。
   * **行情更新检查**：
     * 比较当前 Tick 时间戳与 `_last_tick_time`。
     * **逻辑**：如果 Tick 时间没有更新（旧数据），**返回**。防止同一笔行情重复触发下单，特别是在开盘前后的数据波动期。
4. **下单数量计算**
   * 计算这一笔子单具体下多少手，涉及多重修正逻辑。
   * **基础数量策略**：
     * **固定手数模式**：默认使用配置的 `_order_lots`。
     * **比例模式 (`_by_rate`)**：
       * 取对手盘口量（买入看卖一 `askqty`，卖出看买一 `bidqty`）乘以 `_qty_rate`。
       * 四舍五入，并保证**至少下 1 手**。
   * **修正一：剩余差量封顶**：
     * `this_qty = min(this_qty, abs(diffPos))`。
     * **作用**：确保下单量不会超过剩余需执行的总量。例如剩 3 手，决不能下 10 手。
   * **修正二：平仓封顶（防止反向开仓）**：
     * 获取当前实际持仓 `curPos`。
     * **条件**：如果是买入且持有空单（买平），或者卖出且持有多单（卖平）。
     * **动作**：`this_qty = min(this_qty, abs(curPos))`。
     * **目的**：在平仓阶段，单笔下单量严禁超过现有持仓量。这保证了平仓和反向开仓操作被物理隔离开，避免在同一笔订单中混合操作。
5. **下单价格计算**
   * 根据配置模式计算委托价格，包含追单逻辑。
   * **自动价格模式 (`_price_mode` == 2)**：
     * **盘口压力指标**：计算 `mp = (买一量 - 卖一量) / (买一 + 卖一)`。
     * **趋势判断**：
       * `mp > 0`（买盘强）：买入用卖一价（主动吃），卖出用卖一价（被动挂）。
       * `mp < 0`（卖盘强）：买入用买一价（被动挂），卖出用买一价（主动吃）。
     * **0价修正**：如果取到的价格是 0（可能是数据缺失），回退使用最新价或昨收价。
     * **追单逻辑（核心）**：
       * `buyPx += PriceTick * _cancel_times`
       * `sellPx -= PriceTick * _cancel_times`
       * **逻辑**：利用 `_cancel_times`（撤单次数）作为激进因子。如果上一笔单超时没成交被撤了，说明价格不够有吸引力，这次就加价（买）或降价（卖）。撤单次数越多，价格越激进。
   * **其他模式**：
     * **最优价 (-1)**、**最新价 (0)**、**对手价 (1)**。
     * 在基准价格上应用 `_price_offset`。
   * **通用修正**：
     * 如果计算出的价格为 0，回退使用最新价或昨收价。
6. **涨跌停风控**
   * **修正**：如果买价 > 涨停，修正在涨停价；卖价 < 跌停，修正在跌停价。
   * **不可撤单标记 (`isCanCancel`)**：
     * 如果价格被修正到了涨跌停板，将 `isCanCancel` 设为 `false`。
     * **原因**：在涨跌停板上的挂单通常不需要超时撤单，因为这是能报出的极限价格，撤了再报也一样，不如排队。
7. **执行下单**
   * 根据 `isBuy` 方向调用 `_ctx->buy` 或 `_ctx->sell`。
     * 注意：差量模式下，调用下单接口时 `bForceClose` 参数传 `false`（不强制平仓），因为平仓逻辑已在数量计算中处理。
   * 将返回的订单 ID 加入 `_orders_mon` 进行全生命周期监控。
   * 更新 `_last_place_time` 为当前时间。

```cpp
/**
 * @brief 执行计算实现
 */
void WtDiffMinImpactExeUnit::do_calc()
```

# 订单管理器 WtOrdMon.h/cpp

## 成员
- `IDMap _orders`：订单映射表，存储所有订单的信息（订单ID -> 订单信息对）
  - typedef std::unordered_map\<uint32_t, `OrderPair`\> IDMap：订单映射表类型
    - 订单ID——>订单信息对
  - typedef std::pair<uint64_t, bool> OrderPair：订单信息对类型
    - 订单创建时间（毫秒时间戳）——>是否可撤单（true表示可撤单，false表示不可撤单）
- `tdRecurMutex _mtx_ords`：订单数据的互斥锁，保证线程安全（递归互斥锁，支持同一线程多次加锁）

## 添加订单 push_order
将一批订单 ids[...] 及其对应的当前时间 curTime 和是否可撤单 bCanCancel 存到订单管理器 `_orders`
```cpp
/**
 * @brief 添加订单实现
 * @param ids 订单ID数组指针
 * @param cnt 订单数量
 * @param curTime 当前时间（毫秒时间戳）
 * @param bCanCancel 是否可撤单，true表示可撤单，false表示不可撤单（如涨跌停价的挂单）
 */
void WtOrdMon::push_order(const uint32_t* ids, uint32_t cnt, uint64_t curTime, bool bCanCancel /* = true */)
```

## 删除订单 erase_order
从订单管理器 `_orders` 删除订单ID为 localid 的项
```cpp
/**
 * @brief 删除订单实现
 * @param localid 订单ID
 */
void WtOrdMon::erase_order(uint32_t localid)
```

## 检查是否有订单 has_order
检查订单管理器 `_orders` 是否存在订单ID为localid的订单
```cpp
/**
 * @brief 检查是否有订单
 * @param localid 订单ID，为0时检查是否有任意订单，不为0时检查是否有指定订单（默认0）
 * @return true表示存在订单，false表示不存在订单
 */
inline bool has_order(uint32_t localid = 0)
```

## 检查订单超时 check_orders
检查订单管理器 ` _orders` 中是否有订单超过指定时间未成交，如果有则调用回调函数
- 如果订单不能撤单 !ordInfo.second：跳过
- 如果订单未超时 curTime - ordInfo.first < expiresecs * 1000：跳过
```cpp
/**
 * @brief 检查订单超时实现
 * @param expiresecs 订单超时秒数（订单创建后超过此时间未成交则视为超时）
 * @param curTime 当前时间（毫秒时间戳）
 * @param callback 回调函数，当发现超时订单时调用，参数为订单ID
 */
void WtOrdMon::check_orders(uint32_t expiresecs, uint64_t curTime, EnumOrderCallback callback)
```

## 清空所有订单 clear_orders
```cpp
/**
 * @brief 清空所有订单
 * 清空订单管理器中的所有订单。
 */
inline void clear_orders()
{
    _orders.clear();
}
```

## 枚举所有订单 enumOrder
遍历订单管理器 `_orders` 中的所有订单，对每个订单调用回调函数
```cpp
/**
 * @brief 枚举所有订单实现
 * @param cb 回调函数，对每个订单调用，参数为订单ID、创建时间和可撤单标志
 */
void WtOrdMon::enumOrder(EnumAllOrderCallback cb)
```